In [1]:
!lscpu


Architecture:             x86_64
  CPU op-mode(s):         32-bit, 64-bit
  Address sizes:          46 bits physical, 48 bits virtual
  Byte Order:             Little Endian
CPU(s):                   4
  On-line CPU(s) list:    0-3
Vendor ID:                GenuineIntel
  Model name:             Intel(R) Xeon(R) CPU @ 2.00GHz
    CPU family:           6
    Model:                85
    Thread(s) per core:   2
    Core(s) per socket:   2
    Socket(s):            1
    Stepping:             3
    BogoMIPS:             4000.31
    Flags:                fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge m
                          ca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht sysc
                          all nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xt
                          opology nonstop_tsc cpuid tsc_known_freq pni pclmulqdq
                           ssse3 fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt
                           aes xsave avx f16c rdrand hypervisor 

# Library

In [2]:
# Import các thư viện cần thiết

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, Flatten, Input
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import os
import cv2


import matplotlib.pyplot as plt
import seaborn as sns

# Mining Dataset

# Xử lý outfits

In [3]:
with open('/kaggle/input/vibrent-clothes-rental-dataset/outfits.csv', 'r') as f:
    lines = f.readlines()

# In các dòng đầu tiên để kiểm tra định dạng
for i in range(5):
    print(lines[i])


id;name;description;group;owner;timeCreated;retailPrice;pricePerWeek;pricePerMonth;outfit_tags;tag_categories

outfit.fffdaa715c3646f8b1c0f04d549ff07e;Out of stock - Asymmetric Frilled Dress;This fun, short dress features and asymmetric neckline and an eye catching metallic sheen. Concealed zipper fastening on the side.;group.50a586c78eb7626e294ba3bd07d12c79;o_00053;2017-12-30 11:28:01.000;4000.0000;600.0000;1200.0000;['Synthetic', 'Statement', 'Dresses', 'Metallic', 'Mini', 'Cotton', 'S', 'Black', 'Sandro'];['Material', 'Occasion', 'Category', 'Details', 'Length', 'Material', 'Size', 'Color', 'Brand']

outfit.fffa1b9a3db6415d806f3c48f8ab58d9;Yellow Shell Mellomholmene Blouse;This beautiful blouse features an adjustable neckline, a short and wide silhouette, and ruching details on the sleeves. The cotton fabric makes this the perfect blouse for warmer days. ;group.61ad2fcabb3e9197e3836376e6b67f2c;o_00577;2021-06-07 12:07:22.921;1300.0000;590.0000;1180.0000;['ILAG', 'Tops', 'Spring', 'S

In [4]:
outfits = pd.read_csv('/kaggle/input/vibrent-clothes-rental-dataset/outfits.csv', sep=';', on_bad_lines='skip')


In [5]:
import ast

# Chuyển đổi chuỗi thành danh sách
outfits['outfit_tags'] = outfits['outfit_tags'].apply(ast.literal_eval)
outfits['tag_categories'] = outfits['tag_categories'].apply(ast.literal_eval)


In [6]:
outfits.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15649 entries, 0 to 15648
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              15649 non-null  object 
 1   name            15647 non-null  object 
 2   description     15261 non-null  object 
 3   group           15626 non-null  object 
 4   owner           15649 non-null  object 
 5   timeCreated     15649 non-null  object 
 6   retailPrice     14489 non-null  float64
 7   pricePerWeek    15649 non-null  float64
 8   pricePerMonth   15649 non-null  float64
 9   outfit_tags     15649 non-null  object 
 10  tag_categories  15649 non-null  object 
dtypes: float64(3), object(8)
memory usage: 1.3+ MB


In [7]:
outfits.head()

,id,name,description,group,owner,timeCreated,retailPrice,pricePerWeek,pricePerMonth,outfit_tags,tag_categories
0,outfit.fffdaa715c3646f8b1c0f04d549ff07e,Out of stock - Asymmetric Frilled Dress,"This fun, short dress features and asymmetric ...",group.50a586c78eb7626e294ba3bd07d12c79,o_00053,2017-12-30 11:28:01.000,4000.0,600.0,1200.0,"[Synthetic, Statement, Dresses, Metallic, Mini...","[Material, Occasion, Category, Details, Length..."
1,outfit.fffa1b9a3db6415d806f3c48f8ab58d9,Yellow Shell Mellomholmene Blouse,This beautiful blouse features an adjustable n...,group.61ad2fcabb3e9197e3836376e6b67f2c,o_00577,2021-06-07 12:07:22.921,1300.0,590.0,1180.0,"[ILAG, Tops, Spring, Summer, M, Pattern, Yello...","[Brand, Category, Seasons, Seasons, Size, Deta..."
2,outfit.fff175b13ceb453f9928625491412ede,Kaula Dress Black,Kaula from Rodebjer is a fitted dress made in ...,group.37c2b59d63d3a9c2d58e07f532f71f7f,o_00336,2023-06-05 09:17:59.004,3100.0,930.0,1860.0,"[Black, Mini, M, Everyday, Multi Season, Women...","[Color, Length, Size, Occasion, Seasons, Gende..."
3,outfit.ffef9d7c292a48b69076d2df2e32352f,For sale - Jarvis Blouse,This wrap blouse has mid length sleeves and a ...,group.dfcaa57546b0b7a5e9eb204449b6cc1c,o_00030,2021-05-18 14:02:28.690,1500.0,590.0,1180.0,"[XS, Multi Season, Stylein, Tops, Cotton, Mult...","[Size, Seasons, Brand, Category, Material, Col..."
4,outfit.ffeef842238f4dbdabc6c730a75aa2bd,Black Amber Pants,"Feel slack and nice dressed with this pant, ma...",group.ee297c977905eb21a123a4aea5fbb6d2,o_00602,2021-07-16 14:02:30.643,1200.0,590.0,1180.0,"[Cotton, Black, Everyday, Knitwear, L, Winter,...","[Material, Color, Occasion, Category, Size, Se..."


In [8]:
outfits['outfit_tags'][0][0]

'Synthetic'

In [9]:
outfits['group'][0]

'group.50a586c78eb7626e294ba3bd07d12c79'

In [10]:
outfits.isnull().sum()

id                   0
name                 2
description        388
group               23
owner                0
timeCreated          0
retailPrice       1160
pricePerWeek         0
pricePerMonth        0
outfit_tags          0
tag_categories       0
dtype: int64

In [11]:
print(outfits[['name', 'description']].isnull().sum())


name             2
description    388
dtype: int64


In [12]:
# Lấy các dòng có cột 'name' là null
outfits_null_name = outfits[outfits['name'].isnull()]

# Hiển thị kết quả
outfits_null_name


/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,id,name,description,group,owner,timeCreated,retailPrice,pricePerWeek,pricePerMonth,outfit_tags,tag_categories
10624,outfit.5faafa92b722481ab1414e07380fa463,NaN,NaN,NaN,o_00530,2018-03-22 10:53:01.000,NaN,850.0,1200.0,"[Viscose, Dresses, Reformation, M, Midi, Blue]","[Material, Category, Brand, Size, Length, Color]"
15555,outfit.01b2284d2ea24bfda29e1edc6f2fc0f9,NaN,NaN,group.d2ae7472df69e50536b2f36d37f62861,o_00530,2018-11-26 15:17:49.000,NaN,0.0,0.0,[byTiMo],[Brand]


In [13]:
null_outfit_ids = outfits[outfits['name'].isnull()]['id']
null_outfit_ids

10624    outfit.5faafa92b722481ab1414e07380fa463
15555    outfit.01b2284d2ea24bfda29e1edc6f2fc0f9
Name: id, dtype: object

với cột name loại bỏ missing

In [14]:
outfits = outfits.dropna(subset=['name'])


vơí cột des giữ nguyên thay missing bằng "No description available" nghĩa là không mô tả khả dụng, đảm bảo đầu vào cho mô hình ( tùy trường hợp )

In [15]:
outfits['description'] = outfits['description'].fillna('No description available')


In [16]:
outfits.reset_index(drop=True, inplace=True)  # Đặt lại chỉ mục

In [17]:
outfits.isnull().sum()

id                   0
name                 0
description          0
group               22
owner                0
timeCreated          0
retailPrice       1158
pricePerWeek         0
pricePerMonth        0
outfit_tags          0
tag_categories       0
dtype: int64

In [18]:
outfits.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15647 entries, 0 to 15646
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              15647 non-null  object 
 1   name            15647 non-null  object 
 2   description     15647 non-null  object 
 3   group           15625 non-null  object 
 4   owner           15647 non-null  object 
 5   timeCreated     15647 non-null  object 
 6   retailPrice     14489 non-null  float64
 7   pricePerWeek    15647 non-null  float64
 8   pricePerMonth   15647 non-null  float64
 9   outfit_tags     15647 non-null  object 
 10  tag_categories  15647 non-null  object 
dtypes: float64(3), object(8)
memory usage: 1.3+ MB


In [19]:
outfits.describe()

,retailPrice,pricePerWeek,pricePerMonth
count,14489.000000,15647.000000,15647.000000
mean,2669.603699,671.680769,1314.135361
std,2806.482337,214.003414,461.197713
min,0.000000,0.000000,0.000000
25%,1500.000000,590.000000,1180.000000
50%,2000.000000,590.000000,1180.000000
75%,3000.000000,810.000000,1620.000000
max,80000.000000,5250.000000,10500.000000


In [20]:
outfits.head()

,id,name,description,group,owner,timeCreated,retailPrice,pricePerWeek,pricePerMonth,outfit_tags,tag_categories
0,outfit.fffdaa715c3646f8b1c0f04d549ff07e,Out of stock - Asymmetric Frilled Dress,"This fun, short dress features and asymmetric ...",group.50a586c78eb7626e294ba3bd07d12c79,o_00053,2017-12-30 11:28:01.000,4000.0,600.0,1200.0,"[Synthetic, Statement, Dresses, Metallic, Mini...","[Material, Occasion, Category, Details, Length..."
1,outfit.fffa1b9a3db6415d806f3c48f8ab58d9,Yellow Shell Mellomholmene Blouse,This beautiful blouse features an adjustable n...,group.61ad2fcabb3e9197e3836376e6b67f2c,o_00577,2021-06-07 12:07:22.921,1300.0,590.0,1180.0,"[ILAG, Tops, Spring, Summer, M, Pattern, Yello...","[Brand, Category, Seasons, Seasons, Size, Deta..."
2,outfit.fff175b13ceb453f9928625491412ede,Kaula Dress Black,Kaula from Rodebjer is a fitted dress made in ...,group.37c2b59d63d3a9c2d58e07f532f71f7f,o_00336,2023-06-05 09:17:59.004,3100.0,930.0,1860.0,"[Black, Mini, M, Everyday, Multi Season, Women...","[Color, Length, Size, Occasion, Seasons, Gende..."
3,outfit.ffef9d7c292a48b69076d2df2e32352f,For sale - Jarvis Blouse,This wrap blouse has mid length sleeves and a ...,group.dfcaa57546b0b7a5e9eb204449b6cc1c,o_00030,2021-05-18 14:02:28.690,1500.0,590.0,1180.0,"[XS, Multi Season, Stylein, Tops, Cotton, Mult...","[Size, Seasons, Brand, Category, Material, Col..."
4,outfit.ffeef842238f4dbdabc6c730a75aa2bd,Black Amber Pants,"Feel slack and nice dressed with this pant, ma...",group.ee297c977905eb21a123a4aea5fbb6d2,o_00602,2021-07-16 14:02:30.643,1200.0,590.0,1180.0,"[Cotton, Black, Everyday, Knitwear, L, Winter,...","[Material, Color, Occasion, Category, Size, Se..."


# Xử lý picture_triplets

In [21]:
with open('/kaggle/input/vibrent-clothes-rental-dataset/picture_triplets.csv', 'r') as f:
    lines = f.readlines()

# In các dòng đầu tiên để kiểm tra định dạng
for i in range(5):
    print(lines[i])

picture.id;outfit.id;displayOrder;file_name

picture.0000cdba64314d84a49ed1c266589cc0;outfit.794483397da8425a813301eecf9828c6;0;0000cdba64314d84a49ed1c266589cc0.jpg

picture.00058abb53434872ae9bb4270ae21f8e;outfit.98f32aaf08bc4ff09c44e6e11e9199bc;2;00058abb53434872ae9bb4270ae21f8e.jpg

picture.00063f52c36d43ada95da45f819b30b4;outfit.9fd1c42c3db543c5b6e53b0db1ee8c0f;3;00063f52c36d43ada95da45f819b30b4.jpg

picture.0008443461814f5c988f123718bbd20e;outfit.a7539783b6e94591bdf4e10339afc1d7;3;0008443461814f5c988f123718bbd20e.jpg



In [22]:

pictures= pd.read_csv('/kaggle/input/vibrent-clothes-rental-dataset/picture_triplets.csv' , sep=';', on_bad_lines='skip')



In [23]:
pictures.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50193 entries, 0 to 50192
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   picture.id    50193 non-null  object
 1   outfit.id     50193 non-null  object
 2   displayOrder  50193 non-null  int64 
 3   file_name     50193 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.5+ MB


In [24]:
pictures.head()

,picture.id,outfit.id,displayOrder,file_name
0,picture.0000cdba64314d84a49ed1c266589cc0,outfit.794483397da8425a813301eecf9828c6,0,0000cdba64314d84a49ed1c266589cc0.jpg
1,picture.00058abb53434872ae9bb4270ae21f8e,outfit.98f32aaf08bc4ff09c44e6e11e9199bc,2,00058abb53434872ae9bb4270ae21f8e.jpg
2,picture.00063f52c36d43ada95da45f819b30b4,outfit.9fd1c42c3db543c5b6e53b0db1ee8c0f,3,00063f52c36d43ada95da45f819b30b4.jpg
3,picture.0008443461814f5c988f123718bbd20e,outfit.a7539783b6e94591bdf4e10339afc1d7,3,0008443461814f5c988f123718bbd20e.jpg
4,picture.000a5db3362049aebcc1eb2bf7bde95f,outfit.745fa2bc8156478bac6c0f7d46dadbda,1,000a5db3362049aebcc1eb2bf7bde95f.jpg


In [25]:
null_outfit_ids

10624    outfit.5faafa92b722481ab1414e07380fa463
15555    outfit.01b2284d2ea24bfda29e1edc6f2fc0f9
Name: id, dtype: object

In [26]:
# Bước 2: Loại bỏ các giao dịch liên quan đến các outfit.id đó
pictures_filter = pictures[pictures['outfit.id'].isin(null_outfit_ids)]
pictures_filter

,picture.id,outfit.id,displayOrder,file_name


In [27]:
# Bước 2: Loại bỏ các giao dịch liên quan đến các outfit.id đó
pictures = pictures[~pictures['outfit.id'].isin(null_outfit_ids)]

In [28]:
pictures.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50193 entries, 0 to 50192
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   picture.id    50193 non-null  object
 1   outfit.id     50193 non-null  object
 2   displayOrder  50193 non-null  int64 
 3   file_name     50193 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.5+ MB


# Xử lý user_activity_triplets

In [29]:
with open('/kaggle/input/vibrent-clothes-rental-dataset/user_activity_triplets.csv', 'r') as f:
    lines = f.readlines()

# In các dòng đầu tiên để kiểm tra định dạng
for i in range(5):
    print(lines[i])

customer.id;outfit.id;rentalPeriod.start;rentalPeriod.end

03448;outfit.5c081909537b42239e465d2d615c705f;2023-03-26;2023-04-25

02924;outfit.c34969dd8b334064aa90bfb60c8ec308;2023-03-27;2023-04-26

02924;outfit.aef4cc93eebf40ca8820790deb7a8323;2023-01-27;2023-02-26

01128;outfit.3de5df48a14b4a9aba6d8e41d11e9351;2021-11-16;2021-12-15



In [30]:

transactions = pd.read_csv('/kaggle/input/vibrent-clothes-rental-dataset/user_activity_triplets.csv' , sep=';', on_bad_lines='skip')



In [31]:
transactions['rentalPeriod.start'] = pd.to_datetime(transactions['rentalPeriod.start'])
transactions['rentalPeriod.end'] = pd.to_datetime(transactions['rentalPeriod.end'])


In [32]:
transactions.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64419 entries, 0 to 64418
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer.id         64419 non-null  int64         
 1   outfit.id           64419 non-null  object        
 2   rentalPeriod.start  64419 non-null  datetime64[ns]
 3   rentalPeriod.end    64419 non-null  datetime64[ns]
dtypes: datetime64[ns](2), int64(1), object(1)
memory usage: 2.0+ MB


In [33]:
transactions.head()

,customer.id,outfit.id,rentalPeriod.start,rentalPeriod.end
0,3448,outfit.5c081909537b42239e465d2d615c705f,2023-03-26,2023-04-25
1,2924,outfit.c34969dd8b334064aa90bfb60c8ec308,2023-03-27,2023-04-26
2,2924,outfit.aef4cc93eebf40ca8820790deb7a8323,2023-01-27,2023-02-26
3,1128,outfit.3de5df48a14b4a9aba6d8e41d11e9351,2021-11-16,2021-12-15
4,1128,outfit.0eaa358af14e469894062591bd42f38b,2021-10-11,2021-11-11


In [34]:
null_outfit_ids

10624    outfit.5faafa92b722481ab1414e07380fa463
15555    outfit.01b2284d2ea24bfda29e1edc6f2fc0f9
Name: id, dtype: object

In [35]:
# Bước 2: Loại bỏ các giao dịch liên quan đến các outfit.id đó
transactions_filter = transactions[transactions['outfit.id'].isin(null_outfit_ids)]
transactions_filter

,customer.id,outfit.id,rentalPeriod.start,rentalPeriod.end


In [36]:

# Bước 2: Loại bỏ các giao dịch liên quan đến các outfit.id đó
transactions_cleaned = transactions[~transactions['outfit.id'].isin(null_outfit_ids)]

In [37]:
transactions_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64419 entries, 0 to 64418
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer.id         64419 non-null  int64         
 1   outfit.id           64419 non-null  object        
 2   rentalPeriod.start  64419 non-null  datetime64[ns]
 3   rentalPeriod.end    64419 non-null  datetime64[ns]
dtypes: datetime64[ns](2), int64(1), object(1)
memory usage: 2.0+ MB


# Preprocessing 

In [38]:
outfits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15647 entries, 0 to 15646
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              15647 non-null  object 
 1   name            15647 non-null  object 
 2   description     15647 non-null  object 
 3   group           15625 non-null  object 
 4   owner           15647 non-null  object 
 5   timeCreated     15647 non-null  object 
 6   retailPrice     14489 non-null  float64
 7   pricePerWeek    15647 non-null  float64
 8   pricePerMonth   15647 non-null  float64
 9   outfit_tags     15647 non-null  object 
 10  tag_categories  15647 non-null  object 
dtypes: float64(3), object(8)
memory usage: 1.3+ MB


In [39]:
# Số phần tử không trùng lặp trong cột 'column_name'
unique_count = outfits['id'].nunique()
print(f"Số phần tử không trùng lặpe: {unique_count}")


Số phần tử không trùng lặpe: 15647


In [40]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64419 entries, 0 to 64418
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer.id         64419 non-null  int64         
 1   outfit.id           64419 non-null  object        
 2   rentalPeriod.start  64419 non-null  datetime64[ns]
 3   rentalPeriod.end    64419 non-null  datetime64[ns]
dtypes: datetime64[ns](2), int64(1), object(1)
memory usage: 2.0+ MB


In [41]:
# Số phần tử không trùng lặp trong cột 'column_name'
unique_count = transactions['outfit.id'].nunique()
print(f"Số phần tử không trùng lặp: {unique_count}")


Số phần tử không trùng lặp: 10986


In [42]:
pictures.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50193 entries, 0 to 50192
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   picture.id    50193 non-null  object
 1   outfit.id     50193 non-null  object
 2   displayOrder  50193 non-null  int64 
 3   file_name     50193 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.5+ MB


In [43]:
# Số phần tử không trùng lặp trong cột 'column_name'
unique_count = pictures['outfit.id'].nunique()
print(f"Số phần tử không trùng lặp: {unique_count}")


Số phần tử không trùng lặp: 15157


In [44]:
# Lấy danh sách các outfit.id từ outfits và transactions
outfit_ids = set(outfits['id'])  # Cột 'id' từ outfits
transaction_outfit_ids = set(transactions['outfit.id'])  # Cột 'outfit.id' từ transactions

# Tìm các id có trong transactions nhưng không có trong outfits
foreign_ids = transaction_outfit_ids - outfit_ids

# Kết quả
if foreign_ids:
    print("Các outfit.id ngoại lai (có trong transactions nhưng không có trong outfits):")
    print(foreign_ids)
else:
    print("Không có outfit.id ngoại lai.")


Không có outfit.id ngoại lai.


In [45]:
# Lấy danh sách các outfit.id từ outfits và transactions
pictures_ids = set(pictures['outfit.id'])  # Cột 'id' từ outfits
transaction_outfit_ids = set(transactions['outfit.id'])  # Cột 'outfit.id' từ transactions

# Tìm các id có trong transactions nhưng không có trong outfits
foreign_ids = transaction_outfit_ids - pictures_ids

# Kết quả
if foreign_ids:
    print("Các outfit.id (trong pictures ) ngoại lai (có trong transactions nhưng không có trong outfits):")
    print(foreign_ids)
else:
    print("Không có outfit.id (trong pictures ) ngoại lai.")


Các outfit.id (trong pictures ) ngoại lai (có trong transactions nhưng không có trong outfits):
{'outfit.95a1fba81c574a949ad15d8f18d5027e', 'outfit.53d2ed6b352a419e816708ea2b866bc9', 'outfit.b10df378b77f4ed8acc9c4442b17f924', 'outfit.b41ded69e1bc4182bb558994a9be55fa', 'outfit.acdb17c12e86b6b3', 'outfit.b0a0b74e3e374f95b7b7b41c63b1b5c7'}


In [46]:
transactions = transactions[transactions['outfit.id'].isin(pictures['outfit.id'])]

In [47]:
# Kiểm tra các id có trong transactions nhưng không có trong outfits
foreign_ids_df = transactions[~transactions['outfit.id'].isin(outfits['id'])]

# Kết quả
if not foreign_ids_df.empty:
    print("Các giao dịch có outfit.id không tồn tại trong outfits:")
    print(foreign_ids_df)
else:
    print("Tất cả outfit.id trong transactions đều có trong outfits.")


Tất cả outfit.id trong transactions đều có trong outfits.


In [48]:
# Kiểm tra các id có trong transactions nhưng không có trong pictures
foreign_ids_df = transactions[~transactions['outfit.id'].isin(pictures['outfit.id'])]

# Kết quả
if not foreign_ids_df.empty:
    print("Các giao dịch có outfit.id không tồn tại trong pictures:")
    print(foreign_ids_df)
else:
    print("Tất cả outfit.id trong transactions đều có trong pictures.")

Tất cả outfit.id trong transactions đều có trong pictures.


In [49]:
import time

# Đổi tên cột
data_collect = transactions.rename(columns={
    "outfit.id": "item_id_original",
    "customer.id": "user_id_original",
    "rentalPeriod.start": "time"
})

# Chuyển cột "time" thành UNIX Timestamp (Epoch Time)
data_collect["time"] = pd.to_datetime(data_collect["time"])
data_collect["time"] = data_collect["time"].apply(lambda x: int(time.mktime(x.timetuple())))


data_collect = data_collect[['user_id_original', 'item_id_original' , 'time']]

# Lưu DataFrame đã xử lý thành file CSV mới
output_path = "dataset_VCR.csv"
data_collect.to_csv(output_path, index=False)

# Hiển thị dữ liệu đã xử lý
data_collect.head()


,user_id_original,item_id_original,time
0,3448,outfit.5c081909537b42239e465d2d615c705f,1679788800
1,2924,outfit.c34969dd8b334064aa90bfb60c8ec308,1679875200
2,2924,outfit.aef4cc93eebf40ca8820790deb7a8323,1674777600
3,1128,outfit.3de5df48a14b4a9aba6d8e41d11e9351,1637020800
4,1128,outfit.0eaa358af14e469894062591bd42f38b,1633910400


In [50]:
data_collect.shape

(64413, 3)

In [51]:
data_collect["user_id_original"].nunique()

2293

In [52]:
data_collect["item_id_original"].nunique()

10980

# Xử lý default df

In [53]:
# Đọc file CSV
df = pd.read_csv("/kaggle/working/dataset_VCR.csv")

# Lấy 3 cột chính
df

,user_id_original,item_id_original,time
0,3448,outfit.5c081909537b42239e465d2d615c705f,1679788800
1,2924,outfit.c34969dd8b334064aa90bfb60c8ec308,1679875200
2,2924,outfit.aef4cc93eebf40ca8820790deb7a8323,1674777600
3,1128,outfit.3de5df48a14b4a9aba6d8e41d11e9351,1637020800
4,1128,outfit.0eaa358af14e469894062591bd42f38b,1633910400
...,...,...,...
64408,7376,outfit.5bedabce07614c608b03fbb369d9abe7,1626652800
64409,5837,outfit.bb35c791b2bfa191,1620000000
64410,5837,outfit.ad6f853b5c31b6f5,1620000000
64411,1767,outfit.73393583c562454cbf1c00ce3cac0862,1703721600


In [54]:
df.shape

(64413, 3)

In [55]:
df['user_id_original'].value_counts()

user_id_original
3854    356
6482    348
6425    340
2567    328
617     328
       ... 
1572      1
3395      1
6750      1
1955      1
1155      1
Name: count, Length: 2293, dtype: int64

In [56]:
df['item_id_original'].value_counts()

item_id_original
outfit.925c69db26f6bbae                    38
outfit.aa76777d3b9d4073a342e2093e4ea1e2    34
outfit.a9f2bcb6885fe60a                    32
outfit.9e28084ee8e94a1d8afe4abe643fa8d5    32
outfit.95a9732c69cb1e4c                    31
                                           ..
outfit.c7f52c21ccfb48ef87e304a98182d2d3     1
outfit.b71028a6c2b44ff7806086aa3d1a7ebe     1
outfit.4299225cd0214ba693bb1df3a5640d3f     1
outfit.20edc86dd20b42c0beda4c6ad90f5c3a     1
outfit.092ee39646f34dafac13607ee3c17215     1
Name: count, Length: 10980, dtype: int64

In [57]:


def process_data(df, random_percent=0.1, n_core=10, random_state=42):
    """
    Xử lý dataframe theo quy trình
    1. Lấy random k% số phần tử và lưu lại 
       - random_percent: phần trăm dữ liệu muốn lấy (0-1)
       - random_state: tái tạo kết quả
    2. Áp dụng n-core
       - n_core: số lượng tương tác tối thiểu
    3. Mapping ID cho User và Item
    4. Chia thành train/val/test
    
    """
    
    # 1. Lấy random k% số phần tử
    np.random.seed(random_state)
    n_samples = int(len(df) * random_percent)
    sampled_df = df.sample(n=n_samples, random_state=random_state)
    
    # Lưu file csv với k% phần tử
    # sampled_df.to_csv(f'dataset_VCR_{random_percent}_{random_state}.csv', index=False)
    
    # 2. Áp dụng n-core
    # Đếm số lượng tương tác của mỗi user và item
    user_counts = sampled_df['user_id_original'].value_counts()
    # item_counts = sampled_df['item_id_original'].value_counts()
    
    # Lọc các user và item có ít nhất n tương tác
    valid_users = user_counts[user_counts >= n_core].index
    # valid_items = item_counts[item_counts >= n_core].index
    
    # filtered_df = sampled_df[
    #     sampled_df['user_id_original'].isin(valid_users) & 
    #     sampled_df['item_id_original'].isin(valid_items)
    # ]
    
    # Giữ lại dữ liệu của các user hợp lệ
    filtered_df = sampled_df[sampled_df['user_id_original'].isin(valid_users)]
    
    # 3. Mapping ID
    # Tạo mapping cho user
    unique_users = filtered_df['user_id_original'].unique()
    user_id_map = {old_id: new_id for new_id, old_id in enumerate(unique_users , start=0)}
    
    # Tạo mapping cho item
    unique_items = filtered_df['item_id_original'].unique()
    item_id_map = {old_id: new_id for new_id, old_id in enumerate(unique_items , start=0)}
    
    # Áp dụng mapping
    mapped_df = filtered_df.copy()
    mapped_df['user_id'] = mapped_df['user_id_original'].map(user_id_map)
    mapped_df['item_id'] = mapped_df['item_id_original'].map(item_id_map)

    print(mapped_df.head())
    # Lưu file csv sau mapping
    mapped_df.to_csv(f'dataset_VCR_{random_percent}_{random_state}_{n_core}.csv', index=False)
    
    # # 4. Chia thành train/val/test
    # # Sắp xếp theo thời gian
    # mapped_df = mapped_df.sort_values('time')
    
    # # Chia theo tỷ lệ 70/15/15
    # train_df, temp_df = train_test_split(mapped_df, test_size=0.3, shuffle=False)
    # val_df, test_df = train_test_split(temp_df, test_size=0.5, shuffle=False)
    
    # Lưu các file
    # train_df.to_csv(f'dataset_VCR_{random_percent}_{random_state}_{n_core}_train.csv', index=False)
    # val_df.to_csv(f'dataset_VCR_{random_percent}_{random_state}_{n_core}_val.csv', index=False)
    # test_df.to_csv(f'dataset_VCR_{random_percent}_{random_state}_{n_core}_test.csv', index=False)
    
    # return train_df, val_df, test_df, user_id_map, item_id_map

# Sử dụng hàm
process_data(
    df, 
    random_percent=0.5,  # Lấy 20% dữ liệu
    n_core=10,           # Loại bỏ user/item có ít hơn 5 tương tác
    random_state=42     # Seed để tái tạo kết quả
)

# In thông tin về kết quả
# print(f"Số lượng users sau khi mapping: {len(user_mapping)}")
# print(f"Số lượng items sau khi mapping: {len(item_mapping)}")
# print("\nKích thước các tập dữ liệu:")
# print(f"Train: {len(train)} rows")
# print(f"Validation: {len(val)} rows")
# print(f"Test: {len(test)} rows")

       user_id_original                         item_id_original        time  \
17808              4960  outfit.b3fa81fe8aa241018fe08ca7eeddba28  1683590400   
17992              1625  outfit.be3bd18f74b9420b991a871b4790de26  1704240000   
47089              1557  outfit.557da3a4e43b4aa18835708e7320f359  1638144000   
16521              3605  outfit.b3683ce7eccb4054adbcd73bf6f02d44  1702598400   
28836               849  outfit.4173c7cc17484404a9724ff7465cbef2  1708387200   

       user_id  item_id  
17808        0        0  
17992        1        1  
47089        2        2  
16521        3        3  
28836        4        4  


In [58]:


# def process_data(df, n_core=10, random_state=42):
#     """
#     Xử lý dataframe theo quy trình
#     1. Áp dụng n-core
#        - n_core: số lượng tương tác tối thiểu
#     2. Mapping ID cho User và Item
#     3. Chia thành train/val/test
#     """
    
#     # 1. Áp dụng n-core
#     # Đếm số lượng tương tác của mỗi user
#     user_counts = df['user_id_original'].value_counts()
    
#     # Lọc các user có ít nhất n tương tác
#     valid_users = user_counts[user_counts >= n_core].index
    
#     # Giữ lại dữ liệu của các user hợp lệ
#     filtered_df = df[df['user_id_original'].isin(valid_users)]
    
#     # 2. Mapping ID
#     # Tạo mapping cho user
#     unique_users = filtered_df['user_id_original'].unique()
#     user_id_map = {old_id: new_id for new_id, old_id in enumerate(unique_users, start=0)}
    
#     # Tạo mapping cho item
#     unique_items = filtered_df['item_id_original'].unique()
#     item_id_map = {old_id: new_id for new_id, old_id in enumerate(unique_items, start=0)}
    
#     # Áp dụng mapping
#     mapped_df = filtered_df.copy()
#     mapped_df['user_id'] = mapped_df['user_id_original'].map(user_id_map)
#     mapped_df['item_id'] = mapped_df['item_id_original'].map(item_id_map)

#     print(mapped_df.head())

#     # Sắp xếp theo thời gian
#     mapped_df = mapped_df.sort_values('time')
    
#     # Lưu file csv sau mapping
#     mapped_df.to_csv(f'dataset_VCR_full_{n_core}.csv', index=False)
    
#     # # 3. Chia thành train/val/test
#     # # Sắp xếp theo thời gian
#     # mapped_df = mapped_df.sort_values('time')
    
#     # # Chia theo tỷ lệ 70/15/15
#     # train_df, temp_df = train_test_split(mapped_df, test_size=0.3, shuffle=False)
#     # val_df, test_df = train_test_split(temp_df, test_size=0.5, shuffle=False)
    
#     # # Lưu các file
#     # train_df.to_csv(f'dataset_VCR_full_{n_core}_train.csv', index=False)
#     # val_df.to_csv(f'dataset_VCR_full_{n_core}_val.csv', index=False)
#     # test_df.to_csv(f'dataset_VCR_full_{n_core}_test.csv', index=False)
    
#     # return train_df, val_df, test_df, user_id_map, item_id_map

# # Sử dụng hàm
# process_data(
#     df, 
#     n_core=10,           # Loại bỏ user/item có ít hơn 10 tương tác
#     random_state=42     # Seed để tái tạo kết quả
# )

# # # In thông tin về kết quả
# # print(f"Số lượng users sau khi mapping: {len(user_mapping)}")
# # print(f"Số lượng items sau khi mapping: {len(item_mapping)}")
# # print("\nKích thước các tập dữ liệu:")
# # print(f"Train: {len(train)} rows")
# # print(f"Validation: {len(val)} rows")
# # print(f"Test: {len(test)} rows")

# Xử lý sample df

In [59]:
# Đọc file CSV
df = pd.read_csv("/kaggle/working/dataset_VCR_0.5_42_10.csv")
# df = pd.read_csv("/kaggle/working/dataset_VCR_full_10.csv")

In [60]:
df.head(10)

,user_id_original,item_id_original,time,user_id,item_id
0,4960,outfit.b3fa81fe8aa241018fe08ca7eeddba28,1683590400,0,0
1,1625,outfit.be3bd18f74b9420b991a871b4790de26,1704240000,1,1
2,1557,outfit.557da3a4e43b4aa18835708e7320f359,1638144000,2,2
3,3605,outfit.b3683ce7eccb4054adbcd73bf6f02d44,1702598400,3,3
4,849,outfit.4173c7cc17484404a9724ff7465cbef2,1708387200,4,4
5,6011,outfit.63fc53c3d8784dd39553a33d54ff4921,1696118400,5,5
6,2703,outfit.8069d486b105482394bb94abdc6d10dc,1676505600,6,6
7,3245,outfit.1b3e2ad0256b4c839afe49dcfcd50b40,1667001600,7,7
8,7364,outfit.57ea8cdb48974c93a9e5b65c5711750c,1698796800,8,8
9,96,outfit.d39ea6485a024bb2a6529ce15d2739d7,1707782400,9,9


In [61]:
# df = df[['user_id' , 'item_id' , 'time']]

In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27522 entries, 0 to 27521
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   user_id_original  27522 non-null  int64 
 1   item_id_original  27522 non-null  object
 2   time              27522 non-null  int64 
 3   user_id           27522 non-null  int64 
 4   item_id           27522 non-null  int64 
dtypes: int64(4), object(1)
memory usage: 1.1+ MB


In [63]:
# Sắp xếp dữ liệu theo `user_id` và `time`
df = df.sort_values(by=["user_id", "time"]).reset_index(drop=True)


In [64]:
df.head(10)

,user_id_original,item_id_original,time,user_id,item_id
0,4960,outfit.395fb0e3bb64425787592badb5a34346,1628726400,0,1537
1,4960,outfit.5a117a61fd6e4a2b903d63e961758be7,1631232000,0,1018
2,4960,outfit.20ae7885d88441348ab7a255bcc2173e,1631232000,0,1698
3,4960,outfit.2405f2beaf9241f590e020d163f6fb66,1633651200,0,548
4,4960,outfit.253fb654ace04cbb9280e54d54b62ed8,1633651200,0,4554
5,4960,outfit.f4d096191c4a47d081617a472280443d,1636329600,0,1908
6,4960,outfit.e1bf11e29fe64246b39043783fefa3a4,1636329600,0,3662
7,4960,outfit.801f1b35886e42f6907ec4709420c7e3,1636329600,0,1607
8,4960,outfit.7e8bee4b2c5e4caea0cdcb681b676bd0,1638835200,0,871
9,4960,outfit.652d01e9a0cd459da602c2fd70f558a5,1638835200,0,315


In [65]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27522 entries, 0 to 27521
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   user_id_original  27522 non-null  int64 
 1   item_id_original  27522 non-null  object
 2   time              27522 non-null  int64 
 3   user_id           27522 non-null  int64 
 4   item_id           27522 non-null  int64 
dtypes: int64(4), object(1)
memory usage: 1.1+ MB


# Tạo user list

In [66]:
user_df = df[['user_id_original', 'user_id']]

In [67]:
user_df = user_df.drop_duplicates()

In [68]:

# Sắp xếp theo giá trị user_id tăng dần
user_list_df = user_df.sort_values(by='user_id')

# Ghi vào file user_list.txt, không bao gồm dòng tên cột
user_list_df.to_csv('user_list.txt', sep=' ', index=False, header=False)

print("File user_list.txt đã được tạo.")


File user_list.txt đã được tạo.


In [69]:
user_list_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 909 entries, 0 to 27512
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   user_id_original  909 non-null    int64
 1   user_id           909 non-null    int64
dtypes: int64(2)
memory usage: 21.3 KB


# Tạo item list

In [70]:
item_df = df[['item_id_original', 'item_id']]

In [71]:
item_df = item_df.drop_duplicates()

In [72]:
# Sắp xếp theo giá trị user_id tăng dần
item_list_df = item_df.sort_values(by='item_id')

# Ghi vào file user_list.txt, không bao gồm dòng tên cột
item_list_df.to_csv('item_list.txt', sep=' ', index=False, header=False)

print("File item_list.txt đã được tạo.")

File item_list.txt đã được tạo.


In [73]:
item_list_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8944 entries, 70 to 16476
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   item_id_original  8944 non-null   object
 1   item_id           8944 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 209.6+ KB


In [74]:
item_list_df.head()

,item_id_original,item_id
70,outfit.b3fa81fe8aa241018fe08ca7eeddba28,0
119,outfit.be3bd18f74b9420b991a871b4790de26,1
127,outfit.557da3a4e43b4aa18835708e7320f359,2
226,outfit.b3683ce7eccb4054adbcd73bf6f02d44,3
263,outfit.4173c7cc17484404a9724ff7465cbef2,4


# Tạo interactions matrix

In [75]:
# Tạo dictionary lưu quan hệ user và danh sách item
user_item_interactions = df.groupby('user_id')['item_id'].apply(lambda x: sorted(x)).to_dict()

# Ghi file intersection_user.txt
with open('intersection_user.txt', 'w') as f:
    for user_id in sorted(user_item_interactions.keys()):  # Sắp xếp user_id theo thứ tự tăng dần
        # Lấy danh sách item_id đã được sắp xếp và chuyển thành chuỗi cách nhau bởi khoảng trắng
        item_list_str = ' '.join(map(str, user_item_interactions[user_id]))
        # Ghi user_id và danh sách item vào file
        f.write(f"{user_id} {item_list_str}\n")

print("File intersection_user.txt đã được tạo.")


File intersection_user.txt đã được tạo.


In [76]:
# # Tạo dictionary lưu quan hệ item và danh sách user
# item_user_interactions = df.groupby('item_id')['user_id'].apply(lambda x: sorted(x)).to_dict()

# # Ghi file intersection_item.txt
# with open('intersection_item.txt', 'w') as f:
#     for item_id in sorted(item_user_interactions.keys()):  # Sắp xếp item_id theo thứ tự tăng dần
#         # Lấy danh sách user_id và chuyển thành chuỗi cách nhau bởi khoảng trắng
#         user_list_str = ' '.join(map(str, item_user_interactions[item_id]))  # Sắp xếp user_id
#         # Ghi item_id và danh sách user vào file
#         f.write(f"{item_id} {user_list_str}\n")


## **=> Kiểm tra lại dữ liệu**

In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27522 entries, 0 to 27521
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   user_id_original  27522 non-null  int64 
 1   item_id_original  27522 non-null  object
 2   time              27522 non-null  int64 
 3   user_id           27522 non-null  int64 
 4   item_id           27522 non-null  int64 
dtypes: int64(4), object(1)
memory usage: 1.1+ MB


In [78]:
# Số phần tử không trùng lặp trong cột 'column_name'
unique_count = user_df['user_id_original'].nunique()
print(f"Số phần tử không trùng lặp trong cột user_id_original: {unique_count}")


Số phần tử không trùng lặp trong cột user_id_original: 909


In [79]:
# Số phần tử không trùng lặp trong cột 'column_name'
unique_count = item_df['item_id_original'].nunique()
print(f"Số phần tử không trùng lặp trong cột item_id_original: {unique_count}")


Số phần tử không trùng lặp trong cột item_id_original: 8944


In [80]:
# Số phần tử không trùng lặp trong cột 'column_name'
unique_count = item_df['item_id'].max()
print(f"phần tử lớn nhất trong cột item_id: {unique_count}")

phần tử lớn nhất trong cột item_id: 8943


In [81]:
# Số phần tử không trùng lặp trong cột 'column_name'
unique_count = user_df['user_id'].max()
print(f"phần tử lớn nhất trong cột user_id: {unique_count}")

phần tử lớn nhất trong cột user_id: 908


# Chia train / test 

In [82]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27522 entries, 0 to 27521
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   user_id_original  27522 non-null  int64 
 1   item_id_original  27522 non-null  object
 2   time              27522 non-null  int64 
 3   user_id           27522 non-null  int64 
 4   item_id           27522 non-null  int64 
dtypes: int64(4), object(1)
memory usage: 1.1+ MB


In [83]:
df.tail(10)

,user_id_original,item_id_original,time,user_id,item_id
27512,550,outfit.39e26260f13d466b946601bafd691d36,1599696000,908,927
27513,550,outfit.9a5eec4507c14f97b35b6dd8735dd427,1607558400,908,1162
27514,550,outfit.01601322d21a422fbc2db49c7866e264,1607558400,908,1407
27515,550,outfit.871f15bc73153001,1607558400,908,7258
27516,550,outfit.59a743b5952a415daa0d455a49536872,1671494400,908,6866
27517,550,outfit.3da003afd88d48cc92c189a1d980e358,1671494400,908,8554
27518,550,outfit.27e23087772c4641a0513714504be082,1671494400,908,4975
27519,550,outfit.b3683ce7eccb4054adbcd73bf6f02d44,1674172800,908,3
27520,550,outfit.fc049a65942344bda48d3e5c798a485c,1700438400,908,6167
27521,550,outfit.6a4856dd876644e1aa6a36f8d94cdd39,1703030400,908,783


In [84]:

# Thêm cột `rank` cho từng `user_id`
df["rank"] = df.groupby("user_id").cumcount() + 1


In [85]:
# Tính toán số lượng tương tác và ngưỡng phân chia
user_counts = df.groupby("user_id")["rank"].max().reset_index()
user_counts.rename(columns={"rank": "total_interactions"}, inplace=True)
user_counts["train_threshold"] = (user_counts["total_interactions"] * 0.8).astype(int)


In [86]:
# Gộp ngưỡng với DataFrame ban đầu
df = df.merge(user_counts, on="user_id", how="inner")


In [87]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27522 entries, 0 to 27521
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   user_id_original    27522 non-null  int64 
 1   item_id_original    27522 non-null  object
 2   time                27522 non-null  int64 
 3   user_id             27522 non-null  int64 
 4   item_id             27522 non-null  int64 
 5   rank                27522 non-null  int64 
 6   total_interactions  27522 non-null  int64 
 7   train_threshold     27522 non-null  int64 
dtypes: int64(7), object(1)
memory usage: 1.7+ MB


In [88]:
df.head(10)

,user_id_original,item_id_original,time,user_id,item_id,rank,total_interactions,train_threshold
0,4960,outfit.395fb0e3bb64425787592badb5a34346,1628726400,0,1537,1,101,80
1,4960,outfit.5a117a61fd6e4a2b903d63e961758be7,1631232000,0,1018,2,101,80
2,4960,outfit.20ae7885d88441348ab7a255bcc2173e,1631232000,0,1698,3,101,80
3,4960,outfit.2405f2beaf9241f590e020d163f6fb66,1633651200,0,548,4,101,80
4,4960,outfit.253fb654ace04cbb9280e54d54b62ed8,1633651200,0,4554,5,101,80
5,4960,outfit.f4d096191c4a47d081617a472280443d,1636329600,0,1908,6,101,80
6,4960,outfit.e1bf11e29fe64246b39043783fefa3a4,1636329600,0,3662,7,101,80
7,4960,outfit.801f1b35886e42f6907ec4709420c7e3,1636329600,0,1607,8,101,80
8,4960,outfit.7e8bee4b2c5e4caea0cdcb681b676bd0,1638835200,0,871,9,101,80
9,4960,outfit.652d01e9a0cd459da602c2fd70f558a5,1638835200,0,315,10,101,80


In [89]:
df.tail(10)

,user_id_original,item_id_original,time,user_id,item_id,rank,total_interactions,train_threshold
27512,550,outfit.39e26260f13d466b946601bafd691d36,1599696000,908,927,1,10,8
27513,550,outfit.9a5eec4507c14f97b35b6dd8735dd427,1607558400,908,1162,2,10,8
27514,550,outfit.01601322d21a422fbc2db49c7866e264,1607558400,908,1407,3,10,8
27515,550,outfit.871f15bc73153001,1607558400,908,7258,4,10,8
27516,550,outfit.59a743b5952a415daa0d455a49536872,1671494400,908,6866,5,10,8
27517,550,outfit.3da003afd88d48cc92c189a1d980e358,1671494400,908,8554,6,10,8
27518,550,outfit.27e23087772c4641a0513714504be082,1671494400,908,4975,7,10,8
27519,550,outfit.b3683ce7eccb4054adbcd73bf6f02d44,1674172800,908,3,8,10,8
27520,550,outfit.fc049a65942344bda48d3e5c798a485c,1700438400,908,6167,9,10,8
27521,550,outfit.6a4856dd876644e1aa6a36f8d94cdd39,1703030400,908,783,10,10,8


In [90]:
# Tạo ma trận train và test từ DataFrame
train_interactions = {}
test_interactions = {}

for user_id, user_df in df.groupby('user_id'):
    # Lấy ngưỡng train_threshold cho user hiện tại
    train_threshold = user_df['train_threshold'].iloc[0]

    # Phân chia item_id thành train và test
    train_items = user_df[user_df['rank'] <= train_threshold]['item_id'].tolist()
    test_items = user_df[user_df['rank'] > train_threshold]['item_id'].tolist()

    # Lưu vào dictionary
    train_interactions[user_id] = train_items
    test_interactions[user_id] = test_items


In [91]:
len(train_interactions)

909

In [92]:
len(train_interactions[0])

80

In [93]:
len(test_interactions)

909

In [94]:
len(test_interactions[0])

21

In [95]:
# Ghi file train.txt
with open('train.txt', 'w') as f:
    for user_id, item_list in train_interactions.items():
        item_list_str = ' '.join(map(str, item_list))  # Chuyển danh sách item_id thành chuỗi
        f.write(f"{user_id} {item_list_str}\n")  # Ghi user_id và item_id cách nhau bởi khoảng trắng

# Ghi file test.txt
with open('test.txt', 'w') as f:
    for user_id, item_list in test_interactions.items():
        item_list_str = ' '.join(map(str, item_list))  # Chuyển danh sách item_id thành chuỗi
        f.write(f"{user_id} {item_list_str}\n")  # Ghi user_id và item_id cách nhau bởi khoảng trắng

print("Files train.txt và test.txt đã được tạo.")


Files train.txt và test.txt đã được tạo.


In [96]:
# # Chia dữ liệu
# train_df = df[df["rank"] <= df["train_threshold"]]
# test_df = df[df["rank"] > df["train_threshold"]]


In [97]:
# # Lưu tập train và test thành file
# train_df[["user_id", "item_id"]].to_csv("train.txt", header=False, index=False, sep=" ")
# test_df[["user_id", "item_id"]].to_csv("test.txt", header=False, index=False, sep=" ")

# print("Train and Test data saved to train.txt and test.txt")


=> đang sai chia lại theo hướng , mỗi người dùng , đều có trong train và test (đã fix)

In [98]:
# import pandas as pd
# from sklearn.model_selection import train_test_split

# # Đọc file vào danh sách
# with open("/kaggle/working/intersection.txt", "r") as file:
#     lines = file.readlines()

# # Chia dữ liệu bằng train_test_split
# train, test = train_test_split(lines, test_size=0.2, random_state=42)

# # Lưu kết quả lại thành file
# with open("train.txt", "w") as train_file:
#     train_file.writelines(train)

# with open("test.txt", "w") as test_file:
#     test_file.writelines(test)

# print("Train and Test data saved to train.txt and test.txt")


In [99]:
# len(lines)

In [100]:
# len(train)

In [101]:
# len(test)

# Tạo items_features

## Xử lý feature1 và 2

In [102]:
item_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8944 entries, 0 to 27517
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   item_id_original  8944 non-null   object
 1   item_id           8944 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 209.6+ KB


In [103]:
item_df.reset_index(drop=True, inplace=True)  # Đặt lại chỉ mục

In [104]:
item_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8944 entries, 0 to 8943
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   item_id_original  8944 non-null   object
 1   item_id           8944 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 139.9+ KB


In [105]:
item_df.head()

,item_id_original,item_id
0,outfit.395fb0e3bb64425787592badb5a34346,1537
1,outfit.5a117a61fd6e4a2b903d63e961758be7,1018
2,outfit.20ae7885d88441348ab7a255bcc2173e,1698
3,outfit.2405f2beaf9241f590e020d163f6fb66,548
4,outfit.253fb654ace04cbb9280e54d54b62ed8,4554


In [106]:
outfits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15647 entries, 0 to 15646
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              15647 non-null  object 
 1   name            15647 non-null  object 
 2   description     15647 non-null  object 
 3   group           15625 non-null  object 
 4   owner           15647 non-null  object 
 5   timeCreated     15647 non-null  object 
 6   retailPrice     14489 non-null  float64
 7   pricePerWeek    15647 non-null  float64
 8   pricePerMonth   15647 non-null  float64
 9   outfit_tags     15647 non-null  object 
 10  tag_categories  15647 non-null  object 
dtypes: float64(3), object(8)
memory usage: 1.3+ MB


In [107]:
outfits.head()

,id,name,description,group,owner,timeCreated,retailPrice,pricePerWeek,pricePerMonth,outfit_tags,tag_categories
0,outfit.fffdaa715c3646f8b1c0f04d549ff07e,Out of stock - Asymmetric Frilled Dress,"This fun, short dress features and asymmetric ...",group.50a586c78eb7626e294ba3bd07d12c79,o_00053,2017-12-30 11:28:01.000,4000.0,600.0,1200.0,"[Synthetic, Statement, Dresses, Metallic, Mini...","[Material, Occasion, Category, Details, Length..."
1,outfit.fffa1b9a3db6415d806f3c48f8ab58d9,Yellow Shell Mellomholmene Blouse,This beautiful blouse features an adjustable n...,group.61ad2fcabb3e9197e3836376e6b67f2c,o_00577,2021-06-07 12:07:22.921,1300.0,590.0,1180.0,"[ILAG, Tops, Spring, Summer, M, Pattern, Yello...","[Brand, Category, Seasons, Seasons, Size, Deta..."
2,outfit.fff175b13ceb453f9928625491412ede,Kaula Dress Black,Kaula from Rodebjer is a fitted dress made in ...,group.37c2b59d63d3a9c2d58e07f532f71f7f,o_00336,2023-06-05 09:17:59.004,3100.0,930.0,1860.0,"[Black, Mini, M, Everyday, Multi Season, Women...","[Color, Length, Size, Occasion, Seasons, Gende..."
3,outfit.ffef9d7c292a48b69076d2df2e32352f,For sale - Jarvis Blouse,This wrap blouse has mid length sleeves and a ...,group.dfcaa57546b0b7a5e9eb204449b6cc1c,o_00030,2021-05-18 14:02:28.690,1500.0,590.0,1180.0,"[XS, Multi Season, Stylein, Tops, Cotton, Mult...","[Size, Seasons, Brand, Category, Material, Col..."
4,outfit.ffeef842238f4dbdabc6c730a75aa2bd,Black Amber Pants,"Feel slack and nice dressed with this pant, ma...",group.ee297c977905eb21a123a4aea5fbb6d2,o_00602,2021-07-16 14:02:30.643,1200.0,590.0,1180.0,"[Cotton, Black, Everyday, Knitwear, L, Winter,...","[Material, Color, Occasion, Category, Size, Se..."


In [108]:
item_df['item_id_original'].nunique()

8944

In [109]:
outfits['id'].nunique()

15647

In [110]:
merged_df = pd.merge(outfits, item_df, left_on='id', right_on='item_id_original', how='inner')

In [111]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8944 entries, 0 to 8943
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                8944 non-null   object 
 1   name              8944 non-null   object 
 2   description       8944 non-null   object 
 3   group             8944 non-null   object 
 4   owner             8944 non-null   object 
 5   timeCreated       8944 non-null   object 
 6   retailPrice       8939 non-null   float64
 7   pricePerWeek      8944 non-null   float64
 8   pricePerMonth     8944 non-null   float64
 9   outfit_tags       8944 non-null   object 
 10  tag_categories    8944 non-null   object 
 11  item_id_original  8944 non-null   object 
 12  item_id           8944 non-null   int64  
dtypes: float64(3), int64(1), object(9)
memory usage: 908.5+ KB


In [112]:
merged_df['id'].nunique()

8944

In [113]:
merged_df.head()

,id,name,description,group,owner,timeCreated,retailPrice,pricePerWeek,pricePerMonth,outfit_tags,tag_categories,item_id_original,item_id
0,outfit.fffa1b9a3db6415d806f3c48f8ab58d9,Yellow Shell Mellomholmene Blouse,This beautiful blouse features an adjustable n...,group.61ad2fcabb3e9197e3836376e6b67f2c,o_00577,2021-06-07 12:07:22.921,1300.0,590.0,1180.0,"[ILAG, Tops, Spring, Summer, M, Pattern, Yello...","[Brand, Category, Seasons, Seasons, Size, Deta...",outfit.fffa1b9a3db6415d806f3c48f8ab58d9,2846
1,outfit.ffef9d7c292a48b69076d2df2e32352f,For sale - Jarvis Blouse,This wrap blouse has mid length sleeves and a ...,group.dfcaa57546b0b7a5e9eb204449b6cc1c,o_00030,2021-05-18 14:02:28.690,1500.0,590.0,1180.0,"[XS, Multi Season, Stylein, Tops, Cotton, Mult...","[Size, Seasons, Brand, Category, Material, Col...",outfit.ffef9d7c292a48b69076d2df2e32352f,2712
2,outfit.ffeef842238f4dbdabc6c730a75aa2bd,Black Amber Pants,"Feel slack and nice dressed with this pant, ma...",group.ee297c977905eb21a123a4aea5fbb6d2,o_00602,2021-07-16 14:02:30.643,1200.0,590.0,1180.0,"[Cotton, Black, Everyday, Knitwear, L, Winter,...","[Material, Color, Occasion, Category, Size, Se...",outfit.ffeef842238f4dbdabc6c730a75aa2bd,6747
3,outfit.ffebad2c479045a78adecdcd8f07427d,Bumble Dress Black Burnout Maxi,The Bumble Maxi Dress from Høst & Vår is a ful...,group.10995bb4c49c8a25a48fed9dc025beac,o_00089,2023-01-05 09:12:43.847,1800.0,590.0,1180.0,"[Summer, Dressed-up, Women, S, Spring, Dresses...","[Seasons, Occasion, Gender, Size, Seasons, Cat...",outfit.ffebad2c479045a78adecdcd8f07427d,5871
4,outfit.ffe617f43d244ebc81228915cfcc6c8e,Platinum Blazer,Platinum Blazer is a classic and narrow suit j...,group.f2b0d1fde7052ef61a89b4ad3539b55c,o_00413,2022-04-01 12:25:15.312,2800.0,840.0,1680.0,"[Women, Business, Everyday, Black, Wool, Sprin...","[Gender, Occasion, Occasion, Color, Material, ...",outfit.ffe617f43d244ebc81228915cfcc6c8e,75


In [114]:
merged_df.iloc[0]

id                            outfit.fffa1b9a3db6415d806f3c48f8ab58d9
name                                Yellow Shell Mellomholmene Blouse
description         This beautiful blouse features an adjustable n...
group                          group.61ad2fcabb3e9197e3836376e6b67f2c
owner                                                         o_00577
timeCreated                                   2021-06-07 12:07:22.921
retailPrice                                                    1300.0
pricePerWeek                                                    590.0
pricePerMonth                                                  1180.0
outfit_tags         [ILAG, Tops, Spring, Summer, M, Pattern, Yello...
tag_categories      [Brand, Category, Seasons, Seasons, Size, Deta...
item_id_original              outfit.fffa1b9a3db6415d806f3c48f8ab58d9
item_id                                                          2846
Name: 0, dtype: object

## Xử lý feature3 ( embedings đặc trưng hình ảnh )

In [115]:
pictures.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50193 entries, 0 to 50192
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   picture.id    50193 non-null  object
 1   outfit.id     50193 non-null  object
 2   displayOrder  50193 non-null  int64 
 3   file_name     50193 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.5+ MB


In [116]:
pictures.head()

,picture.id,outfit.id,displayOrder,file_name
0,picture.0000cdba64314d84a49ed1c266589cc0,outfit.794483397da8425a813301eecf9828c6,0,0000cdba64314d84a49ed1c266589cc0.jpg
1,picture.00058abb53434872ae9bb4270ae21f8e,outfit.98f32aaf08bc4ff09c44e6e11e9199bc,2,00058abb53434872ae9bb4270ae21f8e.jpg
2,picture.00063f52c36d43ada95da45f819b30b4,outfit.9fd1c42c3db543c5b6e53b0db1ee8c0f,3,00063f52c36d43ada95da45f819b30b4.jpg
3,picture.0008443461814f5c988f123718bbd20e,outfit.a7539783b6e94591bdf4e10339afc1d7,3,0008443461814f5c988f123718bbd20e.jpg
4,picture.000a5db3362049aebcc1eb2bf7bde95f,outfit.745fa2bc8156478bac6c0f7d46dadbda,1,000a5db3362049aebcc1eb2bf7bde95f.jpg


In [117]:
pictures['outfit.id'].nunique()

15157

In [118]:
print(pictures['outfit.id'].isna().sum())

0


In [119]:
import pandas as pd

# Giả định df là DataFrame của bạn
grouped_sorted_df = (
    pictures.sort_values(by=['outfit.id', 'displayOrder'])  # Sắp xếp theo outfit.id và displayOrder
      .groupby('outfit.id', as_index=False)
      .agg({
          'picture.id': lambda x: list(x),
          'file_name': lambda x: list(x),
          'displayOrder': lambda x: list(x)
      })
)



In [120]:
grouped_sorted_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15157 entries, 0 to 15156
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   outfit.id     15157 non-null  object
 1   picture.id    15157 non-null  object
 2   file_name     15157 non-null  object
 3   displayOrder  15157 non-null  object
dtypes: object(4)
memory usage: 473.8+ KB


In [121]:
grouped_sorted_df.head()

,outfit.id,picture.id,file_name,displayOrder
0,outfit.00004b4d01ca4ab0a70cf073ba74fefa,"[picture.a2b794c7ef83495a8997e7b0c318d65a, pic...","[a2b794c7ef83495a8997e7b0c318d65a.jpg, ba62226...","[1, 2]"
1,outfit.0013691ff35b440e9dcfe1748ec184c7,"[picture.9c821ecbecb14c959f35078010fb91f3, pic...","[9c821ecbecb14c959f35078010fb91f3.jpg, 4d000a0...","[1, 2, 3, 4]"
2,outfit.0014a5c89b244077a3d7cffd4549718e,"[picture.b9aa39eb40f5410fa4fe101236241b19, pic...","[b9aa39eb40f5410fa4fe101236241b19.jpg, d6c3531...","[1, 2, 3]"
3,outfit.0018701ce6b049ebadc314d16623caa8,"[picture.b944a50f20fd4c7f954213dc7c38a776, pic...","[b944a50f20fd4c7f954213dc7c38a776.jpg, e73da8b...","[1, 2, 3]"
4,outfit.001bf665330140cf854dcfb1cbff6b5f,"[picture.21a37e763d1842dca26e7d3276f284c5, pic...","[21a37e763d1842dca26e7d3276f284c5.jpg, 594f077...","[0, 0, 0, 0, 0, 0]"


In [122]:
grouped_sorted_df['outfit.id'].nunique()

15157

In [123]:
grouped_sorted_df.iloc[10000]

outfit.id                 outfit.a92f8950491a4ec893eeef1901ee726e
picture.id      [picture.48eb698319524837bbd8501affb40974, pic...
file_name       [48eb698319524837bbd8501affb40974.jpg, b63c912...
displayOrder                                         [1, 2, 3, 4]
Name: 10000, dtype: object

In [124]:
grouped_sorted_df.iloc[10000]['picture.id']

['picture.48eb698319524837bbd8501affb40974',
 'picture.b63c912ae98d4c2bb7fb07caed8bfaae',
 'picture.9ee12805fc80465f93ab838c9cef336d',
 'picture.e389392f09d743ed9105f6052e4728b7']

In [125]:
grouped_sorted_df.iloc[10000]['picture.id'][0]

'picture.48eb698319524837bbd8501affb40974'

In [126]:
print(merged_df.isna().sum())

id                  0
name                0
description         0
group               0
owner               0
timeCreated         0
retailPrice         5
pricePerWeek        0
pricePerMonth       0
outfit_tags         0
tag_categories      0
item_id_original    0
item_id             0
dtype: int64


In [127]:
print(grouped_sorted_df.isna().sum())

outfit.id       0
picture.id      0
file_name       0
displayOrder    0
dtype: int64


In [128]:
# Kiểm tra các giá trị trong 'id' của merged_df không tồn tại trong 'outfit.id' của grouped_sorted_df
non_matching_ids = merged_df[~merged_df['id'].isin(grouped_sorted_df['outfit.id'])]

# Hiển thị các giá trị không khớp
print(non_matching_ids[['id']])

Empty DataFrame
Columns: [id]
Index: []


In [129]:
import pandas as pd

# Giả định df_a và df_b đã được đọc vào
items_features_df = pd.merge(
    merged_df, grouped_sorted_df,
    left_on='item_id_original',     # Cột tham chiếu từ df_a
    right_on='outfit.id',  # Cột tham chiếu từ df_b
    how='inner'       # Merge theo cách "inner" để giữ lại các hàng khớp
)




In [130]:
items_features_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8944 entries, 0 to 8943
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                8944 non-null   object 
 1   name              8944 non-null   object 
 2   description       8944 non-null   object 
 3   group             8944 non-null   object 
 4   owner             8944 non-null   object 
 5   timeCreated       8944 non-null   object 
 6   retailPrice       8939 non-null   float64
 7   pricePerWeek      8944 non-null   float64
 8   pricePerMonth     8944 non-null   float64
 9   outfit_tags       8944 non-null   object 
 10  tag_categories    8944 non-null   object 
 11  item_id_original  8944 non-null   object 
 12  item_id           8944 non-null   int64  
 13  outfit.id         8944 non-null   object 
 14  picture.id        8944 non-null   object 
 15  file_name         8944 non-null   object 
 16  displayOrder      8944 non-null   object 


In [131]:
print(items_features_df.isna().sum())

id                  0
name                0
description         0
group               0
owner               0
timeCreated         0
retailPrice         5
pricePerWeek        0
pricePerMonth       0
outfit_tags         0
tag_categories      0
item_id_original    0
item_id             0
outfit.id           0
picture.id          0
file_name           0
displayOrder        0
dtype: int64


In [132]:
items_features_df['outfit.id'].nunique()

8944

In [133]:
items_features_df['id'].nunique()

8944

In [134]:
items_features_df.head()

,id,name,description,group,owner,timeCreated,retailPrice,pricePerWeek,pricePerMonth,outfit_tags,tag_categories,item_id_original,item_id,outfit.id,picture.id,file_name,displayOrder
0,outfit.fffa1b9a3db6415d806f3c48f8ab58d9,Yellow Shell Mellomholmene Blouse,This beautiful blouse features an adjustable n...,group.61ad2fcabb3e9197e3836376e6b67f2c,o_00577,2021-06-07 12:07:22.921,1300.0,590.0,1180.0,"[ILAG, Tops, Spring, Summer, M, Pattern, Yello...","[Brand, Category, Seasons, Seasons, Size, Deta...",outfit.fffa1b9a3db6415d806f3c48f8ab58d9,2846,outfit.fffa1b9a3db6415d806f3c48f8ab58d9,"[picture.649ea4f38ffa47eb92556af7d3195ba4, pic...","[649ea4f38ffa47eb92556af7d3195ba4.jpg, fb47724...","[1, 2, 3, 4, 5]"
1,outfit.ffef9d7c292a48b69076d2df2e32352f,For sale - Jarvis Blouse,This wrap blouse has mid length sleeves and a ...,group.dfcaa57546b0b7a5e9eb204449b6cc1c,o_00030,2021-05-18 14:02:28.690,1500.0,590.0,1180.0,"[XS, Multi Season, Stylein, Tops, Cotton, Mult...","[Size, Seasons, Brand, Category, Material, Col...",outfit.ffef9d7c292a48b69076d2df2e32352f,2712,outfit.ffef9d7c292a48b69076d2df2e32352f,"[picture.7d86226f09f040bd81ef18fd24cf4e04, pic...","[7d86226f09f040bd81ef18fd24cf4e04.jpg, e88231a...","[1, 2]"
2,outfit.ffeef842238f4dbdabc6c730a75aa2bd,Black Amber Pants,"Feel slack and nice dressed with this pant, ma...",group.ee297c977905eb21a123a4aea5fbb6d2,o_00602,2021-07-16 14:02:30.643,1200.0,590.0,1180.0,"[Cotton, Black, Everyday, Knitwear, L, Winter,...","[Material, Color, Occasion, Category, Size, Se...",outfit.ffeef842238f4dbdabc6c730a75aa2bd,6747,outfit.ffeef842238f4dbdabc6c730a75aa2bd,"[picture.b9342b85e6a94a6988e37cdb4f3fd0bb, pic...","[b9342b85e6a94a6988e37cdb4f3fd0bb.jpg, 723499a...","[1, 2, 3]"
3,outfit.ffebad2c479045a78adecdcd8f07427d,Bumble Dress Black Burnout Maxi,The Bumble Maxi Dress from Høst & Vår is a ful...,group.10995bb4c49c8a25a48fed9dc025beac,o_00089,2023-01-05 09:12:43.847,1800.0,590.0,1180.0,"[Summer, Dressed-up, Women, S, Spring, Dresses...","[Seasons, Occasion, Gender, Size, Seasons, Cat...",outfit.ffebad2c479045a78adecdcd8f07427d,5871,outfit.ffebad2c479045a78adecdcd8f07427d,"[picture.4a6ba45b29df49949d311ef1709b07fe, pic...","[4a6ba45b29df49949d311ef1709b07fe.jpg, 7c9cb5d...","[1, 2]"
4,outfit.ffe617f43d244ebc81228915cfcc6c8e,Platinum Blazer,Platinum Blazer is a classic and narrow suit j...,group.f2b0d1fde7052ef61a89b4ad3539b55c,o_00413,2022-04-01 12:25:15.312,2800.0,840.0,1680.0,"[Women, Business, Everyday, Black, Wool, Sprin...","[Gender, Occasion, Occasion, Color, Material, ...",outfit.ffe617f43d244ebc81228915cfcc6c8e,75,outfit.ffe617f43d244ebc81228915cfcc6c8e,"[picture.ae0d6fbd73e0496dac7d657c97da7acd, pic...","[ae0d6fbd73e0496dac7d657c97da7acd.jpg, e3bc772...","[1, 2, 3]"


In [135]:
items_features_df.iloc[0]

id                            outfit.fffa1b9a3db6415d806f3c48f8ab58d9
name                                Yellow Shell Mellomholmene Blouse
description         This beautiful blouse features an adjustable n...
group                          group.61ad2fcabb3e9197e3836376e6b67f2c
owner                                                         o_00577
timeCreated                                   2021-06-07 12:07:22.921
retailPrice                                                    1300.0
pricePerWeek                                                    590.0
pricePerMonth                                                  1180.0
outfit_tags         [ILAG, Tops, Spring, Summer, M, Pattern, Yello...
tag_categories      [Brand, Category, Seasons, Seasons, Size, Deta...
item_id_original              outfit.fffa1b9a3db6415d806f3c48f8ab58d9
item_id                                                          2846
outfit.id                     outfit.fffa1b9a3db6415d806f3c48f8ab58d9
picture.id          

In [136]:
# # Thực hiện merge với how='left'
# left_merged_df = pd.merge(
#     merged_df, grouped_sorted_df,
#     left_on='id',     # Cột tham chiếu từ df_a
#     right_on='outfit.id',  # Cột tham chiếu từ df_b
#     how='left'       # Merge theo cách "left" để giữ lại tất cả các hàng từ merged_df
# )

# # Lọc ra các hàng không khớp
# non_matching_rows = left_merged_df[left_merged_df['outfit.id'].isna()]

# # Hiển thị các hàng không khớp
# non_matching_rows.head()

In [137]:
# left_merged_df.info()

## Tạo image_list

In [138]:
image_list_df = items_features_df[['item_id_original' ,'item_id', 'file_name']]

In [139]:
image_list_df.head()

,item_id_original,item_id,file_name
0,outfit.fffa1b9a3db6415d806f3c48f8ab58d9,2846,"[649ea4f38ffa47eb92556af7d3195ba4.jpg, fb47724..."
1,outfit.ffef9d7c292a48b69076d2df2e32352f,2712,"[7d86226f09f040bd81ef18fd24cf4e04.jpg, e88231a..."
2,outfit.ffeef842238f4dbdabc6c730a75aa2bd,6747,"[b9342b85e6a94a6988e37cdb4f3fd0bb.jpg, 723499a..."
3,outfit.ffebad2c479045a78adecdcd8f07427d,5871,"[4a6ba45b29df49949d311ef1709b07fe.jpg, 7c9cb5d..."
4,outfit.ffe617f43d244ebc81228915cfcc6c8e,75,"[ae0d6fbd73e0496dac7d657c97da7acd.jpg, e3bc772..."


In [140]:
image_list_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8944 entries, 0 to 8943
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   item_id_original  8944 non-null   object
 1   item_id           8944 non-null   int64 
 2   file_name         8944 non-null   object
dtypes: int64(1), object(2)
memory usage: 209.8+ KB


In [141]:
# Sắp xếp theo giá trị user_id tăng dần
image_list_df = image_list_df.sort_values(by='item_id')

image_list_df.reset_index(drop=True, inplace=True)  # Đặt lại chỉ mục

In [142]:
image_list_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8944 entries, 0 to 8943
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   item_id_original  8944 non-null   object
 1   item_id           8944 non-null   int64 
 2   file_name         8944 non-null   object
dtypes: int64(1), object(2)
memory usage: 209.8+ KB


In [143]:
image_list_df.head()

,item_id_original,item_id,file_name
0,outfit.b3fa81fe8aa241018fe08ca7eeddba28,0,"[41e17240d3a446bc97224905cee017a0.jpg, 55dced0..."
1,outfit.be3bd18f74b9420b991a871b4790de26,1,"[1ff1010a0fe54cdb857dc729f3c10670.jpg, 5746396..."
2,outfit.557da3a4e43b4aa18835708e7320f359,2,"[8022c15822b741e1b3e7ec38d2c6e7a8.jpg, fedf158..."
3,outfit.b3683ce7eccb4054adbcd73bf6f02d44,3,[9c9b6d38d2084b78a2e99d3f5348957b.jpg]
4,outfit.4173c7cc17484404a9724ff7465cbef2,4,"[5fc2ac1892404c37ab5919714e8b63fa.jpg, 8d64bce..."


In [144]:
# Ghi vào file user_list.txt, không bao gồm dòng tên cột
image_list_df.to_csv('image_list.txt', sep=' ', index=False, header=False)

print("File image_list.txt đã được tạo.")

File image_list.txt đã được tạo.


## Tạo file items_features 

In [145]:
# Tạo cột item_id từ cột index trong df_b
fe_df = items_features_df.copy()

fe_df.head()

,id,name,description,group,owner,timeCreated,retailPrice,pricePerWeek,pricePerMonth,outfit_tags,tag_categories,item_id_original,item_id,outfit.id,picture.id,file_name,displayOrder
0,outfit.fffa1b9a3db6415d806f3c48f8ab58d9,Yellow Shell Mellomholmene Blouse,This beautiful blouse features an adjustable n...,group.61ad2fcabb3e9197e3836376e6b67f2c,o_00577,2021-06-07 12:07:22.921,1300.0,590.0,1180.0,"[ILAG, Tops, Spring, Summer, M, Pattern, Yello...","[Brand, Category, Seasons, Seasons, Size, Deta...",outfit.fffa1b9a3db6415d806f3c48f8ab58d9,2846,outfit.fffa1b9a3db6415d806f3c48f8ab58d9,"[picture.649ea4f38ffa47eb92556af7d3195ba4, pic...","[649ea4f38ffa47eb92556af7d3195ba4.jpg, fb47724...","[1, 2, 3, 4, 5]"
1,outfit.ffef9d7c292a48b69076d2df2e32352f,For sale - Jarvis Blouse,This wrap blouse has mid length sleeves and a ...,group.dfcaa57546b0b7a5e9eb204449b6cc1c,o_00030,2021-05-18 14:02:28.690,1500.0,590.0,1180.0,"[XS, Multi Season, Stylein, Tops, Cotton, Mult...","[Size, Seasons, Brand, Category, Material, Col...",outfit.ffef9d7c292a48b69076d2df2e32352f,2712,outfit.ffef9d7c292a48b69076d2df2e32352f,"[picture.7d86226f09f040bd81ef18fd24cf4e04, pic...","[7d86226f09f040bd81ef18fd24cf4e04.jpg, e88231a...","[1, 2]"
2,outfit.ffeef842238f4dbdabc6c730a75aa2bd,Black Amber Pants,"Feel slack and nice dressed with this pant, ma...",group.ee297c977905eb21a123a4aea5fbb6d2,o_00602,2021-07-16 14:02:30.643,1200.0,590.0,1180.0,"[Cotton, Black, Everyday, Knitwear, L, Winter,...","[Material, Color, Occasion, Category, Size, Se...",outfit.ffeef842238f4dbdabc6c730a75aa2bd,6747,outfit.ffeef842238f4dbdabc6c730a75aa2bd,"[picture.b9342b85e6a94a6988e37cdb4f3fd0bb, pic...","[b9342b85e6a94a6988e37cdb4f3fd0bb.jpg, 723499a...","[1, 2, 3]"
3,outfit.ffebad2c479045a78adecdcd8f07427d,Bumble Dress Black Burnout Maxi,The Bumble Maxi Dress from Høst & Vår is a ful...,group.10995bb4c49c8a25a48fed9dc025beac,o_00089,2023-01-05 09:12:43.847,1800.0,590.0,1180.0,"[Summer, Dressed-up, Women, S, Spring, Dresses...","[Seasons, Occasion, Gender, Size, Seasons, Cat...",outfit.ffebad2c479045a78adecdcd8f07427d,5871,outfit.ffebad2c479045a78adecdcd8f07427d,"[picture.4a6ba45b29df49949d311ef1709b07fe, pic...","[4a6ba45b29df49949d311ef1709b07fe.jpg, 7c9cb5d...","[1, 2]"
4,outfit.ffe617f43d244ebc81228915cfcc6c8e,Platinum Blazer,Platinum Blazer is a classic and narrow suit j...,group.f2b0d1fde7052ef61a89b4ad3539b55c,o_00413,2022-04-01 12:25:15.312,2800.0,840.0,1680.0,"[Women, Business, Everyday, Black, Wool, Sprin...","[Gender, Occasion, Occasion, Color, Material, ...",outfit.ffe617f43d244ebc81228915cfcc6c8e,75,outfit.ffe617f43d244ebc81228915cfcc6c8e,"[picture.ae0d6fbd73e0496dac7d657c97da7acd, pic...","[ae0d6fbd73e0496dac7d657c97da7acd.jpg, e3bc772...","[1, 2, 3]"


## => Tạo feature1 và feature2

In [146]:
# Chuyển 'tag_categories' thành chuỗi, nối các phần tử trong danh sách với khoảng trắng
fe_df['outfit_tags'] = fe_df['outfit_tags'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)

# Tạo cột feature1: kết hợp name và tag_categories với dấu cách giữa chúng
fe_df['feature1'] = fe_df['name'] + ' ' + fe_df['outfit_tags']

# Tạo cột feature2: là cột description từ df_a
fe_df['feature2'] = fe_df['description']

In [147]:
fe_df.head()

,id,name,description,group,owner,timeCreated,retailPrice,pricePerWeek,pricePerMonth,outfit_tags,tag_categories,item_id_original,item_id,outfit.id,picture.id,file_name,displayOrder,feature1,feature2
0,outfit.fffa1b9a3db6415d806f3c48f8ab58d9,Yellow Shell Mellomholmene Blouse,This beautiful blouse features an adjustable n...,group.61ad2fcabb3e9197e3836376e6b67f2c,o_00577,2021-06-07 12:07:22.921,1300.0,590.0,1180.0,ILAG Tops Spring Summer M Pattern Yellow Women...,"[Brand, Category, Seasons, Seasons, Size, Deta...",outfit.fffa1b9a3db6415d806f3c48f8ab58d9,2846,outfit.fffa1b9a3db6415d806f3c48f8ab58d9,"[picture.649ea4f38ffa47eb92556af7d3195ba4, pic...","[649ea4f38ffa47eb92556af7d3195ba4.jpg, fb47724...","[1, 2, 3, 4, 5]",Yellow Shell Mellomholmene Blouse ILAG Tops Sp...,This beautiful blouse features an adjustable n...
1,outfit.ffef9d7c292a48b69076d2df2e32352f,For sale - Jarvis Blouse,This wrap blouse has mid length sleeves and a ...,group.dfcaa57546b0b7a5e9eb204449b6cc1c,o_00030,2021-05-18 14:02:28.690,1500.0,590.0,1180.0,XS Multi Season Stylein Tops Cotton Multicolor...,"[Size, Seasons, Brand, Category, Material, Col...",outfit.ffef9d7c292a48b69076d2df2e32352f,2712,outfit.ffef9d7c292a48b69076d2df2e32352f,"[picture.7d86226f09f040bd81ef18fd24cf4e04, pic...","[7d86226f09f040bd81ef18fd24cf4e04.jpg, e88231a...","[1, 2]",For sale - Jarvis Blouse XS Multi Season Style...,This wrap blouse has mid length sleeves and a ...
2,outfit.ffeef842238f4dbdabc6c730a75aa2bd,Black Amber Pants,"Feel slack and nice dressed with this pant, ma...",group.ee297c977905eb21a123a4aea5fbb6d2,o_00602,2021-07-16 14:02:30.643,1200.0,590.0,1180.0,Cotton Black Everyday Knitwear L Winter Fall T...,"[Material, Color, Occasion, Category, Size, Se...",outfit.ffeef842238f4dbdabc6c730a75aa2bd,6747,outfit.ffeef842238f4dbdabc6c730a75aa2bd,"[picture.b9342b85e6a94a6988e37cdb4f3fd0bb, pic...","[b9342b85e6a94a6988e37cdb4f3fd0bb.jpg, 723499a...","[1, 2, 3]",Black Amber Pants Cotton Black Everyday Knitwe...,"Feel slack and nice dressed with this pant, ma..."
3,outfit.ffebad2c479045a78adecdcd8f07427d,Bumble Dress Black Burnout Maxi,The Bumble Maxi Dress from Høst & Vår is a ful...,group.10995bb4c49c8a25a48fed9dc025beac,o_00089,2023-01-05 09:12:43.847,1800.0,590.0,1180.0,Summer Dressed-up Women S Spring Dresses Flora...,"[Seasons, Occasion, Gender, Size, Seasons, Cat...",outfit.ffebad2c479045a78adecdcd8f07427d,5871,outfit.ffebad2c479045a78adecdcd8f07427d,"[picture.4a6ba45b29df49949d311ef1709b07fe, pic...","[4a6ba45b29df49949d311ef1709b07fe.jpg, 7c9cb5d...","[1, 2]",Bumble Dress Black Burnout Maxi Summer Dressed...,The Bumble Maxi Dress from Høst & Vår is a ful...
4,outfit.ffe617f43d244ebc81228915cfcc6c8e,Platinum Blazer,Platinum Blazer is a classic and narrow suit j...,group.f2b0d1fde7052ef61a89b4ad3539b55c,o_00413,2022-04-01 12:25:15.312,2800.0,840.0,1680.0,Women Business Everyday Black Wool Spring Ricc...,"[Gender, Occasion, Occasion, Color, Material, ...",outfit.ffe617f43d244ebc81228915cfcc6c8e,75,outfit.ffe617f43d244ebc81228915cfcc6c8e,"[picture.ae0d6fbd73e0496dac7d657c97da7acd, pic...","[ae0d6fbd73e0496dac7d657c97da7acd.jpg, e3bc772...","[1, 2, 3]",Platinum Blazer Women Business Everyday Black ...,Platinum Blazer is a classic and narrow suit j...


In [148]:
fe_df.iloc[0]

id                            outfit.fffa1b9a3db6415d806f3c48f8ab58d9
name                                Yellow Shell Mellomholmene Blouse
description         This beautiful blouse features an adjustable n...
group                          group.61ad2fcabb3e9197e3836376e6b67f2c
owner                                                         o_00577
timeCreated                                   2021-06-07 12:07:22.921
retailPrice                                                    1300.0
pricePerWeek                                                    590.0
pricePerMonth                                                  1180.0
outfit_tags         ILAG Tops Spring Summer M Pattern Yellow Women...
tag_categories      [Brand, Category, Seasons, Seasons, Size, Deta...
item_id_original              outfit.fffa1b9a3db6415d806f3c48f8ab58d9
item_id                                                          2846
outfit.id                     outfit.fffa1b9a3db6415d806f3c48f8ab58d9
picture.id          

=> path sẵn có từ dữ liệu gốc ( chưa detect phần thừa )

In [149]:
# import numpy as np
# import os
# # Tạo cột feature3: là danh sách embedings từ df_a

# # Thư mục chứa các file embeddings
# embeddings_dir = '/kaggle/input/vibrent-clothes-rental-dataset/embeddings/EfficientNet_V2_L_final'

# # Hàm tạo danh sách đường dẫn đầy đủ đến tệp .npy
# def create_embedding_paths(picture_ids):
#     return [os.path.join(embeddings_dir, f"{pic_id}.npy") for pic_id in picture_ids]

# # Sử dụng function
# fe_df['embedding_paths'] = fe_df['picture.id'].apply(create_embedding_paths)

=> path mới (đã detect phần thừa lưu ý số chiều )

In [150]:
import numpy as np
import os

# Thư mục chứa các file embeddings
embeddings_dir = '/kaggle/input/mobilenet-emb/embeddings_MBNV2_full'

# Hàm tạo danh sách đường dẫn đầy đủ đến tệp .npy
def create_embedding_paths(picture_ids):
    return [os.path.join(embeddings_dir, f"{pic_id.split('.')[1]}.npy") for pic_id in picture_ids]

# Sử dụng function
fe_df['embedding_paths'] = fe_df['picture.id'].apply(create_embedding_paths)

## combine feature3 embeddings#1 ( mean )

In [151]:
cbf_f31 = fe_df.copy()

In [152]:
cbf_f31.iloc[0]

id                            outfit.fffa1b9a3db6415d806f3c48f8ab58d9
name                                Yellow Shell Mellomholmene Blouse
description         This beautiful blouse features an adjustable n...
group                          group.61ad2fcabb3e9197e3836376e6b67f2c
owner                                                         o_00577
timeCreated                                   2021-06-07 12:07:22.921
retailPrice                                                    1300.0
pricePerWeek                                                    590.0
pricePerMonth                                                  1180.0
outfit_tags         ILAG Tops Spring Summer M Pattern Yellow Women...
tag_categories      [Brand, Category, Seasons, Seasons, Size, Deta...
item_id_original              outfit.fffa1b9a3db6415d806f3c48f8ab58d9
item_id                                                          2846
outfit.id                     outfit.fffa1b9a3db6415d806f3c48f8ab58d9
picture.id          

In [153]:
cbf_f31.iloc[0]['embedding_paths']

['/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/649ea4f38ffa47eb92556af7d3195ba4.npy',
 '/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/fb47724315704dc980a27311bed834e2.npy',
 '/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/c975446e9479495dac2713529edc4230.npy',
 '/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/4685852ed93c439a944ca8ccdd3d1c52.npy',
 '/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/c2d4bb2bf67e4490bdb1c82a9f5bead3.npy']

In [154]:

def load_mean_embeddings(paths):
    """
    Load và combine embeddings của các ảnh trong một outfit
    paths: list các đường dẫn đến embedding files của một outfit
    """
    outfit_embeddings = []
    
    for path in paths:
        if os.path.isfile(path):
            # Load embedding và giữ nguyên shape
            embedding = np.load(path)
            # Print để debug
            # print("Single image embedding shape:", embedding.shape)  # Should be (1280,)
            outfit_embeddings.append(embedding.astype(np.float32))
    
    if outfit_embeddings:
        # Đảm bảo mỗi embedding là 1D array
        outfit_embeddings = [emb.flatten() for emb in outfit_embeddings]
        # Combine embeddings
        combined_embedding = np.mean(outfit_embeddings, axis=0)
        # Print để debug
        # print("Combined embedding shape:", combined_embedding.shape)  # Should be (1280,)
        return combined_embedding.tolist()
    return [0] * 1280  # Return zero vector nếu không có embedding


# Sử dụng function
cbf_f31['feature3'] = cbf_f31['embedding_paths'].apply(load_mean_embeddings)

# Kiểm tra sau khi load
print("Sample of feature3 first element:", len(cbf_f31['feature3'].iloc[0]))  # Should be 1280

Sample of feature3 first element: 1280


## combine feature3 embeddings#2 ( weighted )

In [155]:
cbf_f32 = fe_df.copy()

In [156]:
cbf_f32.iloc[0]

id                            outfit.fffa1b9a3db6415d806f3c48f8ab58d9
name                                Yellow Shell Mellomholmene Blouse
description         This beautiful blouse features an adjustable n...
group                          group.61ad2fcabb3e9197e3836376e6b67f2c
owner                                                         o_00577
timeCreated                                   2021-06-07 12:07:22.921
retailPrice                                                    1300.0
pricePerWeek                                                    590.0
pricePerMonth                                                  1180.0
outfit_tags         ILAG Tops Spring Summer M Pattern Yellow Women...
tag_categories      [Brand, Category, Seasons, Seasons, Size, Deta...
item_id_original              outfit.fffa1b9a3db6415d806f3c48f8ab58d9
item_id                                                          2846
outfit.id                     outfit.fffa1b9a3db6415d806f3c48f8ab58d9
picture.id          

In [157]:
cbf_f32.iloc[0]['embedding_paths']

['/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/649ea4f38ffa47eb92556af7d3195ba4.npy',
 '/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/fb47724315704dc980a27311bed834e2.npy',
 '/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/c975446e9479495dac2713529edc4230.npy',
 '/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/4685852ed93c439a944ca8ccdd3d1c52.npy',
 '/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/c2d4bb2bf67e4490bdb1c82a9f5bead3.npy']

In [158]:

def load_weighted_embeddings(paths, display_orders):
    """
    Load và combine embeddings của các ảnh trong một outfit với trọng số dựa trên display order.
    paths: list các đường dẫn đến embedding files của một outfit.
    display_orders: list thứ tự ưu tiên của các ảnh trong outfit.
    """
    outfit_embeddings = []
    valid_orders = []

    for idx, path in enumerate(paths):
        if os.path.isfile(path):
            embedding = np.load(path).flatten().astype(np.float32)
            outfit_embeddings.append(embedding)
            valid_orders.append(display_orders[idx])

    if outfit_embeddings:
        # Tính trọng số dựa vào displayOrder
        weights = 1 / (np.array(valid_orders) + 1)
        weights = weights / np.sum(weights)  # Chuẩn hóa trọng số
        
        # Áp dụng trọng số vào embeddings
        weighted_embedding = np.sum([w * emb for w, emb in zip(weights, outfit_embeddings)], axis=0)
        
        return weighted_embedding.tolist()

    # Trả về vector zero nếu không có embedding hợp lệ
    return [0] * 1280

# Áp dụng cho DataFrame fe_df
cbf_f32['feature3'] = cbf_f32.apply(
    lambda row: load_weighted_embeddings(row['embedding_paths'], row['displayOrder']),
    axis=1
)

# Kiểm tra kết quả
print("Feature3 sample embedding length:", len(cbf_f32['feature3'].iloc[1998]))  # Should be 1280


Feature3 sample embedding length: 1280


## combine feature3 embeddings#3 ( max )

In [159]:
cbf_f33 = fe_df.copy()

In [160]:
cbf_f33.iloc[0]

id                            outfit.fffa1b9a3db6415d806f3c48f8ab58d9
name                                Yellow Shell Mellomholmene Blouse
description         This beautiful blouse features an adjustable n...
group                          group.61ad2fcabb3e9197e3836376e6b67f2c
owner                                                         o_00577
timeCreated                                   2021-06-07 12:07:22.921
retailPrice                                                    1300.0
pricePerWeek                                                    590.0
pricePerMonth                                                  1180.0
outfit_tags         ILAG Tops Spring Summer M Pattern Yellow Women...
tag_categories      [Brand, Category, Seasons, Seasons, Size, Deta...
item_id_original              outfit.fffa1b9a3db6415d806f3c48f8ab58d9
item_id                                                          2846
outfit.id                     outfit.fffa1b9a3db6415d806f3c48f8ab58d9
picture.id          

In [161]:
cbf_f33.iloc[0]['embedding_paths']

['/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/649ea4f38ffa47eb92556af7d3195ba4.npy',
 '/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/fb47724315704dc980a27311bed834e2.npy',
 '/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/c975446e9479495dac2713529edc4230.npy',
 '/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/4685852ed93c439a944ca8ccdd3d1c52.npy',
 '/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/c2d4bb2bf67e4490bdb1c82a9f5bead3.npy']

In [162]:

def load_max_embeddings(paths):
    """
    Load và combine embeddings của các ảnh trong một outfit
    paths: list các đường dẫn đến embedding files của một outfit
    """
    outfit_embeddings = []
    
    for path in paths:
        if os.path.isfile(path):
            # Load embedding và giữ nguyên shape
            embedding = np.load(path)
            # Print để debug
            # print("Single image embedding shape:", embedding.shape)  # Should be (1280,)
            outfit_embeddings.append(embedding.astype(np.float32))
    
    if outfit_embeddings:
        # Đảm bảo mỗi embedding là 1D array
        outfit_embeddings = [emb.flatten() for emb in outfit_embeddings]
        # Combine embeddings
        combined_embedding = np.max(outfit_embeddings, axis=0)
        # Print để debug
        # print("Combined embedding shape:", combined_embedding.shape)  # Should be (1280,)
        return combined_embedding.tolist()
    return [0] * 1280  # Return zero vector nếu không có embedding


# Sử dụng function
cbf_f33['feature3'] = cbf_f33['embedding_paths'].apply(load_max_embeddings)

# Kiểm tra sau khi load
print("Sample of feature3 first element:", len(cbf_f33['feature3'].iloc[0]))  # Should be 1280

Sample of feature3 first element: 1280


## Dimensionality reduction with PCA

In [163]:
cbf_f31['feature3'].head()

0    [0.1985788643360138, 0.45808306336402893, 0.0,...
1    [1.552445888519287, 0.0, 1.1862998008728027, 1...
2    [0.49159327149391174, 0.96891850233078, 1.6426...
3    [0.21148864924907684, 0.0, 0.1682272106409073,...
4    [0.44339534640312195, 0.0, 0.15799225866794586...
Name: feature3, dtype: object

In [164]:
cbf_f32['feature3'].head()

0    [0.14888234436511993, 0.44530025124549866, 0.0...
1    [1.2419567108154297, 0.0, 0.9671372771263123, ...
2    [0.4549598693847656, 0.70406174659729, 1.73687...
3    [0.2537863850593567, 0.0, 0.1345817744731903, ...
4    [0.30722206830978394, 0.0, 0.14793531596660614...
Name: feature3, dtype: object

In [165]:
cbf_f33['feature3'].head()

0    [0.8535438776016235, 0.7846497297286987, 0.0, ...
1    [3.104891777038574, 0.0, 2.2821121215820312, 2...
2    [0.6065975427627563, 2.641875982284546, 2.1089...
3    [0.4229772984981537, 0.0, 0.3364544212818146, ...
4    [1.326857089996338, 0.0, 0.46035075187683105, ...
Name: feature3, dtype: object

In [166]:
# Gán lại kết quả vào fe_df sau khi kiểm tra
#mean
fe_df['feature3'] = cbf_f31['feature3']
#weight
# fe_df['feature3'] = cbf_f32['feature3']
#max
# fe_df['feature3'] = cbf_f33['feature3']

# Kiểm tra sau khi load
print("Sample of feature3 first element:", len(fe_df['feature3'].iloc[0]))  # Should be 1280

Sample of feature3 first element: 1280


In [167]:
fe_df.iloc[0]

id                            outfit.fffa1b9a3db6415d806f3c48f8ab58d9
name                                Yellow Shell Mellomholmene Blouse
description         This beautiful blouse features an adjustable n...
group                          group.61ad2fcabb3e9197e3836376e6b67f2c
owner                                                         o_00577
timeCreated                                   2021-06-07 12:07:22.921
retailPrice                                                    1300.0
pricePerWeek                                                    590.0
pricePerMonth                                                  1180.0
outfit_tags         ILAG Tops Spring Summer M Pattern Yellow Women...
tag_categories      [Brand, Category, Seasons, Seasons, Size, Deta...
item_id_original              outfit.fffa1b9a3db6415d806f3c48f8ab58d9
item_id                                                          2846
outfit.id                     outfit.fffa1b9a3db6415d806f3c48f8ab58d9
picture.id          

In [168]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
import numpy as np

# Kiểm tra và thay thế NaN bằng vector zero có cùng số chiều
image_embeddings = np.vstack([
    np.array(emb).reshape(1, -1) if isinstance(emb, list) else emb.reshape(1, -1)
    for emb in fe_df['feature3']
])
print("Image embeddings shape before normalization:", image_embeddings.shape)

# Thay thế NaN bằng vector zero
image_embeddings = np.nan_to_num(image_embeddings)

# Chuẩn hóa embeddings trước khi áp dụng PCA
image_embeddings_normalized = normalize(image_embeddings)

# Xác định số chiều mong muốn cho PCA
n_samples, n_features = image_embeddings_normalized.shape
n_components = min(768, n_features, n_samples)

# Áp dụng PCA
pca = PCA(n_components=n_components)
image_embeddings_reduced = pca.fit_transform(image_embeddings_normalized)

# Cập nhật cột feature3 mà không làm thay đổi thứ tự ban đầu
fe_df['feature3'] = list(image_embeddings_reduced)

# Kiểm tra kết quả
print("Feature3 vector size:", len(fe_df['feature3'].iloc[0]))


Image embeddings shape before normalization: (8944, 1280)
Feature3 vector size: 768


In [169]:
print(len(fe_df['feature3'].iloc[0]))  # Kiểm tra vector sau giảm chiều
print("Tổng số mẫu:", len(fe_df))


768
Tổng số mẫu: 8944


In [170]:
fe_df.iloc[0]

id                            outfit.fffa1b9a3db6415d806f3c48f8ab58d9
name                                Yellow Shell Mellomholmene Blouse
description         This beautiful blouse features an adjustable n...
group                          group.61ad2fcabb3e9197e3836376e6b67f2c
owner                                                         o_00577
timeCreated                                   2021-06-07 12:07:22.921
retailPrice                                                    1300.0
pricePerWeek                                                    590.0
pricePerMonth                                                  1180.0
outfit_tags         ILAG Tops Spring Summer M Pattern Yellow Women...
tag_categories      [Brand, Category, Seasons, Seasons, Size, Deta...
item_id_original              outfit.fffa1b9a3db6415d806f3c48f8ab58d9
item_id                                                          2846
outfit.id                     outfit.fffa1b9a3db6415d806f3c48f8ab58d9
picture.id          

In [171]:
fe_df['displayOrder'].head(10)

0    [1, 2, 3, 4, 5]
1             [1, 2]
2          [1, 2, 3]
3             [1, 2]
4          [1, 2, 3]
5                [1]
6                [0]
7       [1, 2, 3, 4]
8          [1, 2, 3]
9       [1, 2, 3, 4]
Name: displayOrder, dtype: object

In [172]:
fe_df['displayOrder'].tail(10)

8934    [1, 2, 3, 4]
8935       [1, 2, 3]
8936    [1, 2, 3, 4]
8937    [0, 0, 0, 0]
8938             [1]
8939    [1, 2, 3, 4]
8940       [1, 2, 3]
8941    [1, 2, 3, 4]
8942    [1, 2, 3, 4]
8943          [1, 2]
Name: displayOrder, dtype: object

In [173]:
fe_df.iloc[0]['embedding_paths']

['/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/649ea4f38ffa47eb92556af7d3195ba4.npy',
 '/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/fb47724315704dc980a27311bed834e2.npy',
 '/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/c975446e9479495dac2713529edc4230.npy',
 '/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/4685852ed93c439a944ca8ccdd3d1c52.npy',
 '/kaggle/input/mobilenet-emb/embeddings_MBNV2_full/c2d4bb2bf67e4490bdb1c82a9f5bead3.npy']

In [174]:
fe_df.iloc[0]['picture.id']

['picture.649ea4f38ffa47eb92556af7d3195ba4',
 'picture.fb47724315704dc980a27311bed834e2',
 'picture.c975446e9479495dac2713529edc4230',
 'picture.4685852ed93c439a944ca8ccdd3d1c52',
 'picture.c2d4bb2bf67e4490bdb1c82a9f5bead3']

In [175]:
len(fe_df.iloc[0]['feature3'])

768

In [176]:
# Chọn chỉ các cột cần thiết để tạo df_c
df_c = fe_df[['item_id', 'feature1', 'feature2' ,'feature3']]

# Hiển thị kết quả
df_c.head()

,item_id,feature1,feature2,feature3
0,2846,Yellow Shell Mellomholmene Blouse ILAG Tops Sp...,This beautiful blouse features an adjustable n...,"[0.10964352284348017, 0.09651304239120508, 0.0..."
1,2712,For sale - Jarvis Blouse XS Multi Season Style...,This wrap blouse has mid length sleeves and a ...,"[-0.13004627777502137, -0.2083090105890317, 0...."
2,6747,Black Amber Pants Cotton Black Everyday Knitwe...,"Feel slack and nice dressed with this pant, ma...","[-0.19392558043462788, -0.1586011855766103, 0...."
3,5871,Bumble Dress Black Burnout Maxi Summer Dressed...,The Bumble Maxi Dress from Høst & Vår is a ful...,"[-0.35813668018597444, -0.018954081450289352, ..."
4,75,Platinum Blazer Women Business Everyday Black ...,Platinum Blazer is a classic and narrow suit j...,"[-0.4170340325049188, -0.16846395736268407, -0..."


In [177]:
df_c['item_id'][0]

2846

In [178]:
df_c.iloc[0]

item_id                                                  2846
feature1    Yellow Shell Mellomholmene Blouse ILAG Tops Sp...
feature2    This beautiful blouse features an adjustable n...
feature3    [0.10964352284348017, 0.09651304239120508, 0.0...
Name: 0, dtype: object

In [179]:
df_c['feature1'][0]

'Yellow Shell Mellomholmene Blouse ILAG Tops Spring Summer M Pattern Yellow Women Blouses Everyday Cotton'

In [180]:
df_c['feature2'][0]

'This beautiful blouse features an adjustable neckline, a short and wide silhouette, and ruching details on the sleeves. The cotton fabric makes this the perfect blouse for warmer days. '

In [181]:
df_c['feature3'][1998][0]

0.1755824442470876

In [182]:
df_c.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8944 entries, 0 to 8943
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   item_id   8944 non-null   int64 
 1   feature1  8944 non-null   object
 2   feature2  8944 non-null   object
 3   feature3  8944 non-null   object
dtypes: int64(1), object(3)
memory usage: 279.6+ KB


In [183]:
df_c.isnull().sum()

item_id     0
feature1    0
feature2    0
feature3    0
dtype: int64

In [184]:
# Lưu DataFrame df_c vào file CSV
df_c.to_csv('items_features.csv', index=False)

# Hiển thị thông báo lưu thành công
print("DataFrame đã được lưu vào items_features.csv")


DataFrame đã được lưu vào items_features.csv


# Debug Item feature (sửa khi load items_features trong load_data)

In [185]:
xxx = pd.read_csv('/kaggle/working/items_features.csv')

In [186]:
xxx.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8944 entries, 0 to 8943
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   item_id   8944 non-null   int64 
 1   feature1  8944 non-null   object
 2   feature2  8944 non-null   object
 3   feature3  8944 non-null   object
dtypes: int64(1), object(3)
memory usage: 279.6+ KB


In [187]:
xxx.head()

,item_id,feature1,feature2,feature3
0,2846,Yellow Shell Mellomholmene Blouse ILAG Tops Sp...,This beautiful blouse features an adjustable n...,[ 1.09643523e-01 9.65130424e-02 8.44811590e-...
1,2712,For sale - Jarvis Blouse XS Multi Season Style...,This wrap blouse has mid length sleeves and a ...,[-1.30046278e-01 -2.08309011e-01 7.91625547e-...
2,6747,Black Amber Pants Cotton Black Everyday Knitwe...,"Feel slack and nice dressed with this pant, ma...",[-1.93925580e-01 -1.58601186e-01 1.79685231e-...
3,5871,Bumble Dress Black Burnout Maxi Summer Dressed...,The Bumble Maxi Dress from Høst & Vår is a ful...,[-3.58136680e-01 -1.89540815e-02 -1.35791300e-...
4,75,Platinum Blazer Women Business Everyday Black ...,Platinum Blazer is a classic and narrow suit j...,[-4.17034033e-01 -1.68463957e-01 -1.12388731e-...


In [188]:
xxx['feature3'][0]

'[ 1.09643523e-01  9.65130424e-02  8.44811590e-02 -1.86984879e-02\n -1.09450685e-01  1.57179867e-01  1.20255944e-01 -1.46143766e-01\n -9.74966429e-03  1.13435408e-01 -1.16760295e-02  2.91654603e-02\n -4.34674899e-02 -1.24051679e-01  4.79845393e-03 -7.93851824e-02\n -2.73531832e-02 -5.75467683e-02  3.55287791e-02 -4.61642157e-02\n -4.53384866e-02 -5.15398766e-02 -5.34103725e-02  3.58137725e-02\n  1.59658213e-02 -1.58451934e-02 -2.63646160e-02 -2.93628023e-02\n -6.52932002e-02  1.17712693e-02  4.21524681e-02 -6.74482470e-02\n -6.17109343e-03 -7.00347933e-02 -7.65909480e-03  6.25067573e-02\n -4.74212383e-03 -5.48204285e-02  4.29739546e-02  1.52694513e-02\n -2.87110627e-02  2.30776753e-02 -9.41021669e-03 -8.21608884e-03\n  2.93901950e-02  2.55357349e-02 -5.84884700e-03  1.89265239e-02\n  2.06973889e-02 -1.19271445e-02 -6.40919987e-02 -4.67790165e-02\n -1.09818095e-02 -1.78093350e-02  2.20604100e-02 -4.30548764e-02\n -3.31586766e-02 -1.94563732e-02  4.98048852e-02 -2.41408310e-02\n -1.54471

In [189]:
xxx['feature3'][0][0]

'['

In [190]:
# Chuyển string vector thành numpy array
def parse_vector_string(vector_string):
    # Loại bỏ dấu ngoặc vuông và split theo khoảng trắng
    vector = vector_string.strip('[]').split()
    # Chuyển đổi sang float
    return np.array([float(x) for x in vector])

image_embeddings = np.vstack(
    xxx['feature3'].apply(parse_vector_string).values
)

In [191]:
image_embeddings[0]

array([ 1.09643523e-01,  9.65130424e-02,  8.44811590e-02, -1.86984879e-02,
       -1.09450685e-01,  1.57179867e-01,  1.20255944e-01, -1.46143766e-01,
       -9.74966429e-03,  1.13435408e-01, -1.16760295e-02,  2.91654603e-02,
       -4.34674899e-02, -1.24051679e-01,  4.79845393e-03, -7.93851824e-02,
       -2.73531832e-02, -5.75467683e-02,  3.55287791e-02, -4.61642157e-02,
       -4.53384866e-02, -5.15398766e-02, -5.34103725e-02,  3.58137725e-02,
        1.59658213e-02, -1.58451934e-02, -2.63646160e-02, -2.93628023e-02,
       -6.52932002e-02,  1.17712693e-02,  4.21524681e-02, -6.74482470e-02,
       -6.17109343e-03, -7.00347933e-02, -7.65909480e-03,  6.25067573e-02,
       -4.74212383e-03, -5.48204285e-02,  4.29739546e-02,  1.52694513e-02,
       -2.87110627e-02,  2.30776753e-02, -9.41021669e-03, -8.21608884e-03,
        2.93901950e-02,  2.55357349e-02, -5.84884700e-03,  1.89265239e-02,
        2.06973889e-02, -1.19271445e-02, -6.40919987e-02, -4.67790165e-02,
       -1.09818095e-02, -

In [192]:
len(image_embeddings)

8944

In [193]:
len(image_embeddings[0])

768

## Debug embeddings 

In [194]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from transformers import BertModel, BertTokenizer

In [195]:
def preprocess_text(text):
    text = text.lower()
    return text

In [196]:
xxx['combined_text'] = xxx['feature1'] + ' ' + xxx['feature2']
xxx['cleaned_combined_text'] = xxx['combined_text'].apply(preprocess_text)

# Tạo BERT embeddings cho văn bản
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [197]:
import torch

In [198]:
def get_bert_embeddings(text):
    inputs = tokenizer(text, return_tensors='pt', max_length=512, truncation=True, padding='max_length')
    with torch.no_grad():
        outputs = model(**inputs)
    # BERT base có kích thước embedding là 768
    embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
    return embeddings

# text_embeddings = np.vstack(
#     xxx['cleaned_combined_text'].apply(lambda x: get_bert_embeddings(x)).to_numpy()
# )

In [199]:
text_embeddings = np.vstack(get_bert_embeddings(xxx['cleaned_combined_text'][0]))

In [200]:
text_embeddings

array([[-1.32130712e-01, -4.80578125e-01,  2.61906356e-01,
         4.98509035e-04,  1.92617998e-02,  2.40128532e-01,
         1.13349840e-01,  1.76416248e-01, -1.37547314e-01,
        -3.16068172e-01, -9.78496224e-02, -1.42556384e-01,
        -1.66068539e-01,  5.95610365e-02, -1.39088720e-01,
         2.75807027e-02,  1.24800183e-01,  3.01254928e-01,
        -1.00319728e-01,  2.92990118e-01,  1.33612260e-01,
        -1.59170210e-01,  9.03117061e-02,  1.17086917e-01,
         4.02323544e-01, -5.27881086e-02, -1.23920990e-03,
         1.43134380e-02,  1.62785321e-01,  2.64486820e-01,
         1.20635331e-03, -6.07602671e-03,  1.51898682e-01,
        -1.17829554e-01,  4.28457148e-02, -3.87182057e-01,
        -1.99187919e-02, -3.18495594e-02, -4.16335613e-02,
        -9.95824933e-02, -2.50402868e-01, -6.39423504e-02,
         2.24446714e-01, -2.71887362e-01, -1.92225426e-01,
        -4.29721892e-01, -1.66247189e-01,  2.64995873e-01,
        -4.76105124e-01, -4.20513004e-02, -3.83637011e-0

In [201]:
# Chuyển đổi về float32
text_embeddings = text_embeddings.astype(np.float32)
image_embeddings = image_embeddings.astype(np.float32)


In [202]:
# Kiểm tra kiểu dữ liệu (dtype), kích thước (shape), và số chiều (ndim)
print("Text Embeddings:")
print("Type:", type(text_embeddings))
print("Data type (dtype):", text_embeddings.dtype)
print("Shape:", text_embeddings.shape)
print("Number of dimensions (ndim):", text_embeddings.ndim)

print("\nImage Embeddings:")
print("Type:", type(image_embeddings))
print("Data type (dtype):", image_embeddings.dtype)
print("Shape:", image_embeddings.shape)
print("Number of dimensions (ndim):", image_embeddings.ndim)


Text Embeddings:
Type: <class 'numpy.ndarray'>
Data type (dtype): float32
Shape: (1, 768)
Number of dimensions (ndim): 2

Image Embeddings:
Type: <class 'numpy.ndarray'>
Data type (dtype): float32
Shape: (8944, 768)
Number of dimensions (ndim): 2
